<a href="https://colab.research.google.com/github/AliBsrnglu/Central-Bank---Text-Analysis/blob/main/NLTK%2BZEMBEREK%2BLEX%C4%B0CON(1_7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
!rm -rf /content/*
print("🚀 Colab çalışma alanı tamamen sıfırlandı.")

🚀 Colab çalışma alanı tamamen sıfırlandı.


In [2]:
# @title
# ===============================================
# 🧱 COLAB HÜCRE 1 — ORTAM & BAĞIMLILIKLAR (SAFE KEEP)
# ===============================================

!pip install regex --quiet


# (Opsiyonel) Hızlı Zemberek smoke testini açmak istersen True yap.
SMOKE_TEST = True

# Temel kütüphaneler (veri işleme + regex + görselleştirme)
!pip install pandas matplotlib nltk --quiet

# Java-Python köprüsü (Zemberek için zorunlu)
!pip install jpype1 --quiet

# Zemberek Python wrapper (Java ile konuşmamızı sağlar)
!pip install zemberek-python --quiet

# Gerekiyorsa: Zemberek’in jar dosyasını indir (resmi kaynak)
import os
from pathlib import Path
JAR_PATH = Path("zemberek-full.jar")
if not JAR_PATH.exists():
    !wget -q -O zemberek-full.jar \
      https://github.com/ahmetaa/zemberek-nlp/releases/download/v0.17.1/zemberek-full.jar

# ===============================================
# 🧠 NLTK İLK KURULUM (idempotent)
# ===============================================
import nltk

def _ensure_nltk_resources():
    needed = {
        "punkt": "tokenizers/punkt",
        "stopwords": "corpora/stopwords",
    }
    for pkg, res in needed.items():
        try:
            nltk.data.find(res)
        except LookupError:
            nltk.download(pkg, quiet=True)

_ensure_nltk_resources()

# ===============================================
# ✅ ORTAM TESTİ
# ===============================================
import jpype
import pandas as pd
import re
import matplotlib.pyplot as plt

print("Python ortamı hazır.")
print("NLTK, JPype, Pandas, Matplotlib kurulu.")
print(f"Zemberek JAR mevcut mu? {JAR_PATH.exists()} — yol: {JAR_PATH.resolve()}")

# (OPSİYONEL) Kısa smoke test — davranışı değiştirmemek için varsayılan False
if SMOKE_TEST:
    try:
        if not jpype.isJVMStarted():
            import jpype
            from jpype import getDefaultJVMPath
            jpype.startJVM(
                getDefaultJVMPath(),
                "-ea",
                f"-Djava.class.path={str(JAR_PATH)}",
                convertStrings=True
            )
        from jpype import JClass
        TurkishMorphology = JClass("zemberek.morphology.TurkishMorphology")
        morph = TurkishMorphology.createWithDefaults()
        analysis = morph.analyze("enflasyonun").getAnalyzes()
        print("Zemberek smoke test (kök aday sayısı):", analysis.size())
    except Exception as e:
        print("⚠️ Smoke test atlandı:", e)
    finally:
        # JVM'i bu hücrede açık bırakmıyoruz; asıl akışını değiştirmemek için kapatıyoruz.
        try:
            if jpype.isJVMStarted():
                jpype.shutdownJVM()
        except Exception:
            pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 MB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.8 which is incompatible.
Python ortamı hazır.
NLTK, JPype, Pandas, Matplotlib kurulu.
Zemberek JAR mevcut mu? True — yol: /content/zemberek-full.jar
⚠️ Smoke test atlandı: Class zemberek.morphology.TurkishMorphology is not found


In [3]:
# @title
# ===============================================
# 🧱 COLAB HÜCRE 2 — SÖZLÜK + REGEX DOĞRULAMA (UNICODE/SAFE)
# ===============================================

import os, io, csv, hashlib, unicodedata, warnings
import pandas as pd
from pathlib import Path
from google.colab import files

# (Opsiyonel) Zemberek hazırsa dursun; değilse sorun değil
try:
    import jpype
    from jpype import JClass
    if not jpype.isJVMStarted():
        JAR_PATH = Path("/content/zemberek-full.jar")
        if JAR_PATH.exists():
            jpype.startJVM("-Xmx2G", classpath=[str(JAR_PATH)], convertStrings=True)
    TurkishMorphology = JClass("zemberek.morphology.TurkishMorphology")
    morph = TurkishMorphology.createWithDefaults()
except Exception:
    morph = None

# --- Regex motoru: varsa 'regex', yoksa 're' ---
try:
    import regex as _re   # \p{L}, \P{L}, VERSION1, timeout
    _HAS_REGEX = True
except Exception:
    import re as _re
    _HAS_REGEX = False

# ---------- Yardımcılar ----------

def _nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s) if isinstance(s, str) else ""

# Etiket + desen karışımı gelirse ilk gerçek metakarakterden itibaren regex'i al,
# hiç metakarakter yoksa tamamını literal kabul et (escape et).
_META = _re.compile(r"(\(\?|\(|\[|\\|\||\^|\$|\?|\+|\*|\{)")
def sanitize_pattern(pat: str) -> str:
    if not isinstance(pat, str):
        return ""
    s = _nfc(pat.strip())
    m = _META.search(s)
    if m:
        return s[m.start():].strip()
    return _re.escape(s)

_FLAGS = _re.IGNORECASE | (_re.VERSION1 if _HAS_REGEX else 0)

def compile_regex(pattern: str):
    pat = sanitize_pattern(pattern)
    if not pat:
        return None
    try:
        return _re.compile(_nfc(pat), _FLAGS)
    except Exception as e:
        warnings.warn(f"Hatalı Regex: {pattern}  →  {e}")
        return None

# ============ 0) Bellek temizliği ============
for var in ["df_dict", "REGEX_CACHE", "CURRENT_DICT_FINGERPRINT"]:
    if var in globals():
        del globals()[var]
REGEX_CACHE = {}
CURRENT_DICT_FINGERPRINT = None

# ============ 1) Güvenli yükleme ============
def upload_fresh_file(expected_prefix="Dictionary", encoding="utf-8-sig"):
    # klasördeki eski sözlükleri sil
    for f in os.listdir():
        if expected_prefix.lower() in f.lower():
            try:
                os.remove(f)
            except Exception as e:
                print(f"⚠️ Silinemedi: {f} ({e})")
    print(f"🧹 Eski {expected_prefix} dosyaları temizlendi.")

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Dosya yüklenmedi.")
    file_name = list(uploaded.keys())[0]
    print(f"📂 Yüklendi: {file_name}")

    # Önyüz JS çöpü varsa başlığa kadar kırp
    clean_name = file_name
    with open(file_name, "r", encoding=encoding, errors="ignore") as f:
        lines = f.readlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith(("Terim", "terim", "TERİM")):
            header_idx = i
            break
    if header_idx not in (None, 0):
        clean_name = file_name.replace(".txt", "_CLEAN.txt").replace(".csv", "_CLEAN.csv")
        with open(clean_name, "w", encoding=encoding) as f:
            f.writelines(lines[header_idx:])
        print(f"🧼 Colab karışımı temizlendi → {clean_name}")

    with open(clean_name, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    return clean_name, digest

# ============ 2) Ayırıcı tespiti ve okuma ============
def sniff_and_read(path, encoding="utf-8-sig"):
    sample = Path(path).read_text(encoding=encoding, errors="ignore")[:4096]
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";\t,")
        sep = dialect.delimiter
    except Exception:
        first = sample.splitlines()[0] if sample else ""
        if ";" in first:
            sep = ";"
        elif "\t" in first:
            sep = "\t"
        elif "," in first:
            sep = ","
        else:
            sep = ";"
    df = pd.read_csv(path, sep=sep, encoding=encoding, engine="python")
    return df, sep

# ============ 3) Sütun eşleme ve tip güvenliği ============
def normalize_columns(df: pd.DataFrame):
    df = df.rename(columns={c: c.strip() for c in df.columns})

    regex_col = None
    for cand in ["Regex/Patern", "Regex", "Desen", "Pattern"]:
        if cand in df.columns:
            regex_col = cand
            break
    if regex_col is None:
        raise ValueError("Sözlükte 'Regex/Patern' (veya 'Regex') sütunu zorunludur.")

    if "Polarite" not in df.columns:
        df["Polarite"] = 0
    df["Polarite"] = pd.to_numeric(df["Polarite"], errors="coerce").fillna(0).astype(float)

    if "Politika" not in df.columns:
        df["Politika"] = ""
    df["Politika"] = df["Politika"].fillna("").astype(str)

    for opt in ["Terim", "Tür", "Kavram", "Kaynak"]:
        if opt not in df.columns:
            df[opt] = ""
        df[opt] = df[opt].fillna("").astype(str)

    df = df[df[regex_col].astype(str).str.strip() != ""].copy()
    df.reset_index(drop=True, inplace=True)
    return df, regex_col

# ============ 4) Sözlük doğrulama ve duman testi ============
def validate_dictionary(df: pd.DataFrame, regex_col: str, sample_texts=None, max_show=8):
    df = df.copy()
    df[regex_col] = df[regex_col].astype(str).apply(sanitize_pattern)
    df["CompiledRegex"] = df[regex_col].apply(compile_regex)

    bad = df["CompiledRegex"].isna()
    if bad.any():
        print(f"⚠️ Derlenemeyen desen sayısı: {bad.sum()}")
        display(df.loc[bad, [regex_col]].head(20))

    ok_df = df.loc[~bad].copy()
    print(f"✅ Derlenen desen sayısı: {len(ok_df)}")

    if sample_texts:
        print("\n🔎 Örnek metinlerde eşleşme duman testi:")
        rows = []
        for i, row in ok_df.head(200).iterrows():
            rx = row["CompiledRegex"]
            hit = None
            txt_idx = None
            for k, t in enumerate(sample_texts):
                try:
                    m = rx.search(t) if not _HAS_REGEX else rx.search(t, timeout=0.02)
                except Exception:
                    m = None
                if m:
                    hit, txt_idx = m.group(0), k
                    break
            rows.append({
                "idx": i,
                "Terim": row.get("Terim", ""),
                "Regex": row[regex_col],
                "Match": hit or "",
                "Text#": txt_idx if hit else "",
                "Polarite": row.get("Polarite", ""),
                "Politika": row.get("Politika", "")
            })
        rep = pd.DataFrame(rows)
        show = rep[rep["Match"] != ""].head(max_show)
        display(show if not show.empty else pd.DataFrame([{"Bilgi": "İlk 200 desende örnek metin eşleşmesi yok."}]))
    return ok_df

# ============ 5) Polarite↔Kavram kayması düzeltmesi ============
def fix_polarity_shift(df: pd.DataFrame) -> pd.DataFrame:
    if "Polarite" not in df.columns or "Kavram" not in df.columns:
        return df
    pol_num_ratio = pd.to_numeric(df["Polarite"], errors="coerce").notna().mean()
    kav_num_ratio = pd.to_numeric(df["Kavram"],  errors="coerce").notna().mean()
    if pol_num_ratio < 0.2 and kav_num_ratio > 0.8:
        df = df.copy()
        tmp = df["Polarite"].astype(str)
        df["Polarite"] = pd.to_numeric(df["Kavram"], errors="coerce").fillna(0).astype(float)
        df["Kavram"]   = tmp
        print("🔧 Polarite↔Kavram sütun kayması otomatik düzeltildi.")
    return df

# ============ 6) Çalıştır ============
file_name, CURRENT_DICT_FINGERPRINT = upload_fresh_file("Dictionary")
df_raw, sep_used = sniff_and_read(file_name)
print(f"📑 Ayırıcı: '{sep_used}' | Ham satır: {len(df_raw)}")

df_raw = fix_polarity_shift(df_raw)
df_dict, regex_col = normalize_columns(df_raw)
print(f"🧱 Sütunlar: {list(df_dict.columns)}")
print(f"🎯 Regex sütunu: {regex_col} | Kayıt: {len(df_dict)}")

# Sözlük regex kolonunu sterilize et (etiket/literal koruması)
df_dict[regex_col] = df_dict[regex_col].astype(str).apply(sanitize_pattern)

# Örnek cümlelerle duman testi
sample_texts = [
    "Küresel para politikalarına dair belirsizlikler ve büyümeye ilişkin endişeler nedeniyle finansal oynaklık artmıştır.",
    "İç talepte ılımlı büyüme sürerken finansal istikrar güçlenmiştir; kur oynaklığında artış gözlenmiştir.",
    "Faiz oynaklıklarında azalış gözlenmiş, sıkı para politikası duruşu korunmuştur."
]
df_ok = validate_dictionary(df_dict, regex_col, sample_texts=sample_texts, max_show=10)

print("\n✅ Sözlük ve regex doğrulaması tamamlandı. Yeni yüklenen sözlük belleğe işlendi.")
print("Polarite dağılımı:", df_ok["Polarite"].value_counts(dropna=False).to_dict())

# =========================
# 📊 Sözlük Tablo Görünümü
# =========================
try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except Exception:
    pass

_cols = ["Terim", "Polarite", "Tür", "Kavram", "Politika", regex_col, "Kaynak"]
cols_show = [c for c in _cols if c in df_ok.columns]
df_view = df_ok.loc[:, cols_show].copy()
display(df_view)

# (Opsiyonel) Renklendirme
def _pol_color(v):
    try:
        v = float(v)
        if v < 0: return "color:#B00020; font-weight:600"
        if v > 0: return "color:#0B8043; font-weight:600"
        return "color:#5F6368"
    except Exception:
        return ""

# Eski:
# styled = (df_view
#           .style
#           .applymap(_pol_color, subset=["Polarite"])
#           .set_properties(**{"white-space": "pre"}))

# Yeni:
styled = (df_view.style
          .map(_pol_color, subset=["Polarite"])
          .set_properties(**{"white-space": "pre"}))
# display(styled)  # İstersen aç


🧹 Eski Dictionary dosyaları temizlendi.


Saving Dictionary.txt to Dictionary.txt
📂 Yüklendi: Dictionary.txt
📑 Ayırıcı: ';' | Ham satır: 21
🧱 Sütunlar: ['Terim', 'Polarite', 'Kavram', 'Tür', 'Politika', 'Regex/Patern', 'Kaynak']
🎯 Regex sütunu: Regex/Patern | Kayıt: 21
✅ Derlenen desen sayısı: 21

🔎 Örnek metinlerde eşleşme duman testi:


,idx,Terim,Regex,Match,Text#,Polarite,Politika
0,0,belirsizlik,(?<!\w)belirsiz[\wçğıöşüÇĞİÖŞÜ]*(?!\w),belirsizlikler,0,-1.0,
1,1,endişe,(?<!\w)endişe[\wçğıöşüÇĞİÖŞÜ]*(?!\w),endişeler,0,-1.0,
2,2,oynaklık,(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?!\w),oynaklık,0,-1.0,
5,5,istikrar,(?<!\w)istikrar[\wçğıöşüÇĞİÖŞÜ]*(?!\w),istikrar,1,1.0,
8,8,faiz oynaklıklarında azalış,"(?<!\w)faiz[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,10}?\...",Faiz oynaklıklarında azalış,2,1.0,
17,17,oynaklık azalış,"(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,10}?...",oynaklıklarında azalış,2,1.0,



✅ Sözlük ve regex doğrulaması tamamlandı. Yeni yüklenen sözlük belleğe işlendi.
Polarite dağılımı: {-1.0: 11, 1.0: 9, 0.0: 1}


,Terim,Polarite,Tür,Kavram,Politika,Regex/Patern,Kaynak
0,belirsizlik,-1.0,unigram,belirsizlik,,(?<!\w)belirsiz[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
1,endişe,-1.0,unigram,endişe,,(?<!\w)endişe[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
2,oynaklık,-1.0,unigram,oynaklık,,(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
3,denge,1.0,unigram,denge,,(?<!\w)denge(?!siz)[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
4,risk,-1.0,unigram,risk,,(?<!\w)risk[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
5,istikrar,1.0,unigram,istikrar,,(?<!\w)istikrar[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
6,iyileşme,1.0,unigram,iyileşme,,(?<!\w)iyile[şş]?[\wçğıöşüÇĞİÖŞÜ]*(?!\w),enf-ocak2016_tam
7,finansal oynaklık devamı,-1.0,trigram,finans oynaklık devam,,"(?<!\w)finans[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,8}?...",enf-ocak2016_tam
8,faiz oynaklıklarında azalış,1.0,trigram,faiz oynaklık azalış,,"(?<!\w)faiz[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,10}?\...",enf-ocak2016_tam
9,finansal oynaklık artışı,-1.0,trigram,finans oynaklık artış,,"(?<!\w)finans[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,8}?...",enf-ocak2016_tam


In [4]:
# @title
# ===============================================
# 📂 COLAB HÜCRE 3 — Dual Layer Text Preprocessing (STABLE/CLEAN v2)
# ===============================================

from google.colab import files
import io, re, csv, unicodedata, string
import pandas as pd
import nltk

# ---- NLTK kaynaklarını koşullu indir (idempotent) ----
def _ensure_nltk():
    need = {
        "punkt": "tokenizers/punkt",
        "stopwords": "corpora/stopwords",
    }
    for pkg, res in need.items():
        try:
            nltk.data.find(res)
        except LookupError:
            nltk.download(pkg, quiet=True)
    # Opsiyonel 'punkt_tab' desteği (yeni NLTK'lerde var)
    try:
        nltk.data.find("tokenizers/punkt_tab")
    except LookupError:
        try:
            nltk.download("punkt_tab", quiet=True)
        except Exception:
            pass

_ensure_nltk()

from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

# ---- Türkçe stopwords ----
stop_words = set(stopwords.words("turkish"))

# ---- Türkçe güvenli küçük harfe çevirme + Unicode normalize ----
def tr_lower(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = unicodedata.normalize("NFKC", text)
    t = t.replace("İ", "i").replace("I", "ı")
    t = t.lower()
    return unicodedata.normalize("NFC", t)

# ---- CSV ayırıcı kestirimi (;, \t, ,) ----
def sniff_csv_delimiter(sample_bytes: bytes) -> str:
    try:
        sample_txt = sample_bytes.decode("utf-8-sig", errors="ignore")[:4096]
        dialect = csv.Sniffer().sniff(sample_txt, delimiters=";\t,")
        return dialect.delimiter
    except Exception:
        first = sample_txt.splitlines()[0] if sample_txt else ""
        if ";" in first: return ";"
        if "\t" in first: return "\t"
        if "," in first: return ","
        return ","

# ---- Noktalama seti (Unicode genişletilmiş) ----
_EXTRA_PUNCT = "“”‘’—–…·•«»%₺€−"
_PUNCT_TABLE = str.maketrans({c: " " for c in (string.punctuation + _EXTRA_PUNCT)})

def strip_punct(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text.translate(_PUNCT_TABLE)
    t = re.sub(r"[^\S\r\n]+", " ", t)
    return t

# ---- Güvenli tokenizasyon ----
def _safe_word_tokenize(text: str):
    try:
        # Cümleleyiciye ihtiyaç duymadan tokenize et (punkt_tab bağımlılığı yok)
        return word_tokenize(text, language="turkish", preserve_line=True)
    except Exception:
        # Regex fallback: yalnızca harf tokenları (Türkçe dahil), min 3 harf
        return re.findall(r"[A-Za-zÇĞİÖŞÜçğıöşü]{3,}", text)

def clean_sentence(sentence: str, keep_min_len: int = 3):
    s = tr_lower(sentence)
    s = strip_punct(s)
    tokens = _safe_word_tokenize(s)
    tokens = [
        w for w in tokens
        if w not in stop_words and len(w) >= keep_min_len and w.isalpha()
    ]
    return " ".join(tokens)

print("🔸 Lütfen analiz etmek istediğin TCMB raporunu yükle (.txt veya .csv)...")
uploaded_report = files.upload()
if not uploaded_report:
    raise ValueError("❌ Hiç dosya seçilmedi. Lütfen bir TCMB rapor dosyası yükle.")

report_file_name = list(uploaded_report.keys())[0]
print(f"📄 Yüklenen rapor dosyası: {report_file_name}")

# ========== Dosya okuma ==========
if report_file_name.lower().endswith(".csv"):
    raw = uploaded_report[report_file_name]
    delimiter = sniff_csv_delimiter(raw)
    df_report = pd.read_csv(io.BytesIO(raw), sep=delimiter, encoding="utf-8-sig", engine="python")
    text_cols = [c for c in df_report.columns if df_report[c].dtype == "object"]
    if not text_cols:
        text_series = df_report.astype(str).agg(" ".join, axis=1)
    else:
        text_series = df_report[text_cols].astype(str).agg(" ".join, axis=1)
    text_data = " ".join(text_series.tolist())
elif report_file_name.lower().endswith(".txt"):
    text_data = io.TextIOWrapper(
        io.BytesIO(uploaded_report[report_file_name]),
        encoding="utf-8-sig",
        errors="ignore"
    ).read()
else:
    raise ValueError("❌ Desteklenmeyen dosya biçimi. Lütfen .txt veya .csv yükle.")

# ========== 1) Cümle Segmentasyonu ==========
try:
    sentences = sent_tokenize(text_data, language="turkish")
except Exception:
    sentences = re.split(r"(?<=[\.\!\?])\s+|\n{2,}", text_data)
    sentences = [s for s in sentences if s and not s.isspace()]

print(f"📘 Cümle segmentasyonu tamamlandı. Toplam {len(sentences)} cümle bulundu.\n")

# ========== 2) Cümle Temizleme ==========
rows = []
for i, s in enumerate(sentences, start=1):
    s_clean = clean_sentence(s)
    rows.append({
        "Cümle_ID": i,
        "Cümle_Orijinal": s.strip(),
        "Cümle_Temiz": s_clean,
        "Kelime_Sayısı": len(s_clean.split()) if s_clean else 0
    })

df_clean = pd.DataFrame(rows)

print("✅ Çift katmanlı temizleme tamamlandı (orijinal + analiz metni).")
display(df_clean.head(20)),


🔸 Lütfen analiz etmek istediğin TCMB raporunu yükle (.txt veya .csv)...


Saving Test.txt to Test.txt
📄 Yüklenen rapor dosyası: Test.txt
📘 Cümle segmentasyonu tamamlandı. Toplam 12 cümle bulundu.

✅ Çift katmanlı temizleme tamamlandı (orijinal + analiz metni).


,Cümle_ID,Cümle_Orijinal,Cümle_Temiz,Kelime_Sayısı
0,1,Küresel para politikalarına dair belirsizlikle...,küresel para politikalarına dair belirsizlikle...,15
1,2,2015 yılı Aralık ayında ABD Merkez Bankası (Fe...,yılı aralık ayında abd merkez bankası fed nın ...,38
2,3,Küresel iktisadi faaliyette\n2014 yılından ber...,küresel iktisadi faaliyette yılından beri yaşa...,18
3,4,Emtia fiyatları da yakın dönemde düşüş eğilimi...,emtia fiyatları yakın dönemde düşüş eğilimini ...,8
4,5,Gelişmekte olan ülkeler bu dönemde küresel dal...,gelişmekte olan ülkeler dönemde küresel dalgal...,9
5,6,Bu ülkelere\nyönelik portföy akımları zayıf gö...,ülkelere yönelik portföy akımları zayıf görünü...,12
6,7,Küresel piyasalarda yaşanan oynaklığın etkiler...,küresel piyasalarda yaşanan oynaklığın etkiler...,8
7,8,Bununla\nbirlikte yurt içi belirsizliklerin az...,bununla birlikte yurt içi belirsizliklerin aza...,24
8,9,Milli gelir ılımlı büyüme eğilimini istikrarlı...,milli gelir ılımlı büyüme eğilimini istikrarlı...,9
9,10,Artan jeopolitik risklere\nkarşın Avrupa Birli...,artan jeopolitik risklere karşın avrupa birliğ...,14


(None,)

In [5]:
# @title
# ===============================================
# 🧮 COLAB HÜCRE 4 — Cümle Bazlı Eşleşme + Skor
# REVIZE v3.2 — hijyen + CUSTOM_TRANSFORM kancası
# Tür önceliği: trigram > bigram > unigram  (örtüşme çözümünde garanti)
# df_hits_full = HAM SAYAÇ (örtüşme çözümü YOK, tekilleştirme YOK)
# df_hits      = TEKILLEŞTIRILMIŞ (2-C) + Tür Öncelikli Örtüşme (3-B+TypeRank)
# 4-B normalizasyon: sum / (1 + ln n)
# Politika tonu (daraltıcı/genişletici): skordan tamamen hariç
# SAFE→FALLBACK taraması + teşhis kolonları
# ===============================================

import math, unicodedata, warnings
import pandas as pd
from typing import List, Dict, Any, Tuple
from IPython.display import display             # hijyen: görüntüleme için
import re as RE                                  # hijyen: temel regex modülü

# ── Ayarlar ──────────────────────────────────────────────────────────────────
POLICY_TONE_NEUTRALIZE = True   # politika tonunu skordan tamamen çıkar
DEFAULT_GAP12 = 8
DEFAULT_GAP23 = 12
CONTEXT_CHARS = 80
BYPASS_ANCHOR_PREFILTER = True  # False yapılırsa basit anchor filtresi devreye girer
MIN_ANCHOR_LEN = 3

# Örtüşme çözümünde tür önceliğini etkinleştirir:
TYPE_PRIORITY_STRICT = True     # trigram > bigram > unigram

# ── Regex motoru ─────────────────────────────────────────────────────────────
try:
    import regex as RX
    _HAS_REGEX = True
except Exception:
    RX = None
    _HAS_REGEX = False

_ENGINE = RX if _HAS_REGEX else RE
_FLAGS  = _ENGINE.IGNORECASE | (_ENGINE.VERSION1 if _HAS_REGEX else 0)
_RX_TIMEOUT = 0.10  # sn

# ── Unicode sınıfları ────────────────────────────────────────────────────────
if _HAS_REGEX:
    L  = r"\p{L}"
    nL = r"\P{L}"
else:
    L  = r"[^\W\d_]"
    nL = r"[^\w]"

LB = rf"(?<!{L})"
RB = rf"(?!{L})"

def WORD(root: str) -> str:
    root = root or ""
    return rf"{root}{L}*"

def GAP(gap: int) -> str:
    return rf"(?:{nL}+{L}+)" + rf"{{0,{gap}}}" + rf"{nL}+"

# ── Çıkış placeholder'ları ───────────────────────────────────────────────────
expected_cols = ["Cümle_ID","start","end","Eşleşme","Bağlam",
                 "Terim","Tür","Kavram","Politika","Regex",
                 "Polarite_Base","Polarite_Final","Regex_Kaynağı"]

df_hits            = pd.DataFrame(columns=expected_cols)
df_hits_full       = pd.DataFrame(columns=expected_cols)
df_sentence_scores = pd.DataFrame(columns=["Cümle_ID","Cümle_Orijinal","Cümle_Temiz",
                                           "Eşleşme_Sayısı","Eşleşme_Sayısı_Skor",
                                           "Cümle_Skor_RawSum","Cümle_Skor"])
ts_summary_df      = pd.DataFrame(columns=["pos","neg","Net Oran","Pozitif Pay","Net/1000 kelime","Toplam kelime"])
diag_df            = pd.DataFrame(columns=["Cümle_ID","Anchor_Pass","Regex_Tried","Raw_Matches","Cümle_Önizleme"])

# ── Yardımcılar ──────────────────────────────────────────────────────────────
def _tr_lower_norm(t: str) -> str:
    if not isinstance(t, str):
        return ""
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("İ","i").replace("I","ı").lower()
    return unicodedata.normalize("NFC", t)

def _norm_tr(s):
    if s is None: return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("İ","i").replace("I","ı").lower()
    return unicodedata.normalize("NFC", s).strip()

# Politika bucket — hem skor hariç bırakma hem rapor için tek kaynak
def _policy_bucket_local(v: str) -> str:
    t = _norm_tr(v)
    if not t: return ""
    if t == "daraltıcı" or "daralt" in t or "sıkı" in t or "sıkılaş" in t:
        return "daraltıcı"
    if t == "genişletici" or "geniş" in t or "gevş" in t:
        return "genişletici"
    return ""

def _compile_local(pat: str):
    if not isinstance(pat, str) or not pat.strip():
        return None
    try:
        return _ENGINE.compile(unicodedata.normalize("NFC", pat), _FLAGS)
    except Exception as e:
        warnings.warn(f"Hatalı Regex: {pat} -> {e}")
        return None

def _expand_c_token(c: str) -> str:
    c = _tr_lower_norm(c)
    if c.startswith("art"):
        return r"art(?:ış|an|tı|ti|ıyor|mak|maktadır|makta)?"
    if c.startswith("yüksel"):
        return r"yüksel(?:iş|en|di|iyor|mek|mektedir|mekte)?"
    if c.startswith("azal"):
        return r"azal(?:ış|ma|an|dı|iyor|mak|mektedir|mekte)?"
    if c.startswith("devam"):
        return r"devam"
    return _tr_lower_norm(c)

def _safe_int(x, default: int) -> int:
    try:
        v = int(float(str(x).strip()))
        return max(0, v)
    except Exception:
        return default

# Sözlük satırından güvenli regex üretimi
def _safe_regex_from_row(row) -> str:
    tur = str(row.get("Tür","")).strip().lower()
    src = row.get("Kavram") or row.get("Terim") or ""
    toks = [t for t in _tr_lower_norm(str(src)).split() if t]

    gap12 = _safe_int(row.get("Gap12", DEFAULT_GAP12), DEFAULT_GAP12)
    gap23 = _safe_int(row.get("Gap23", DEFAULT_GAP23), DEFAULT_GAP23)

    if tur == "unigram":
        if toks:
            a = toks[0]
            return rf"{LB}{WORD(a)}{RB}"
        else:
            base = row.get("Regex") or row.get("Regex/Patern") or ""
            pieces = RE.findall(r"[A-Za-zÇĞİÖŞÜçğıöşü]{3,}", str(base))
            a = pieces[0].lower() if pieces else ""
            return rf"{LB}{WORD(a)}{RB}" if a else ""
    elif tur == "bigram":
        if len(toks) < 2: return ""
        a, b = toks[0], toks[1]
        return rf"{LB}{WORD(a)}{GAP(gap12)}{WORD(b)}{RB}"
    elif tur == "trigram":
        if len(toks) < 3: return ""
        a, b, c = toks[0], toks[1], toks[2]
        c_pat = _expand_c_token(c)
        return rf"{LB}{WORD(a)}{GAP(gap12)}{WORD(b)}{GAP(gap23)}(?:{c_pat}){L}*{RB}"
    else:
        if toks:
            return rf"{LB}{WORD(toks[0])}{RB}"
        return ""

# Timeout güvenli finditer
try:
    _RegexPatternType = type(RX.compile("a")) if _HAS_REGEX else ()
except Exception:
    _RegexPatternType = ()
_RePatternType = type(RE.compile("a"))

def _iter_finditer(rxpat, text: str):
    if rxpat is None or not isinstance(text, str):
        return
    if _HAS_REGEX and isinstance(rxpat, _RegexPatternType):
        try:
            for m in rxpat.finditer(text, timeout=_RX_TIMEOUT):
                yield m
        except Exception:
            return
    elif isinstance(rxpat, _RePatternType):
        for m in rxpat.finditer(text):
            yield m

def _signed_unit(p) -> float:
    try:
        fp = float(p)
    except Exception:
        return 0.0
    return 1.0 if fp > 0 else (-1.0 if fp < 0 else 0.0)

def _row_polarity(row) -> float:
    """Politika tonda 0.0; aksi halde Polarite'nin işaretli birimi."""
    if POLICY_TONE_NEUTRALIZE:
        if _policy_bucket_local(row.get("Politika","")) in {"daraltıcı","genişletici"}:
            return 0.0
    return _signed_unit(row.get("Polarite", 0.0))

# Tür önceliği (trigram < bigram < unigram) — sıralama anahtarı için
def _type_rank(t: str) -> int:
    t = (t or "").strip().lower()
    if not TYPE_PRIORITY_STRICT:
        return 0  # kapalıysa etkisiz
    if t.startswith("tri"): return 0
    if t.startswith("bi"):  return 1
    if t.startswith("uni"): return 2
    return 3

# 3-B(+TypeRank): Örtüşme çözümü — tür önceliği garanti + diğer kriterler
def _resolve_overlaps_priority(hits: List[Dict[str,Any]]) -> List[int]:
    """
    Sıralama:
      1) IsPolicyTone=False önce
      2) |polarite| büyük olan önce
      3) Tür önceliği: trigram > bigram > unigram
      4) Daha uzun span önce
      5) Daha erken başlangıç önce
    """
    order = sorted(
        range(len(hits)),
        key=lambda i: (
            hits[i].get("IsPolicyTone", False),                 # False (0) önce
            -float(hits[i].get("AbsPol", 0.0)),                 # 1.0 önce
            _type_rank(hits[i].get("Tür","")),                  # tür önceliği
            -(int(hits[i]["end"]) - int(hits[i]["start"])),     # uzunluk
            int(hits[i]["start"]),
            i
        )
    )
    kept_idx, occupied = [], []
    for i in order:
        s, e = int(hits[i]["start"]), int(hits[i]["end"])
        overlap = any(not (e <= s2 or e2 <= s) for (s2, e2) in occupied)
        if overlap:
            continue
        kept_idx.append(i); occupied.append((s, e))
    return sorted(kept_idx)

# 2-C: Aynı cümlede (Terim, Regex) başına tek oy
def _dedup_term_regex(hits_sent: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen, kept = set(), []
    for h in hits_sent:
        key = (h.get("Terim",""), h.get("Regex",""))
        if key in seen:
            continue
        seen.add(key); kept.append(h)
    return kept

# ── Ön koşul ─────────────────────────────────────────────────────────────────
if "df_ok" not in globals() or "df_clean" not in globals():
    print("ℹ️ Bu motor hücresi df_ok (Hücre-2) ve df_clean (Hücre-3) olmadan çalışmaz.")
    print("   Yine de boş DataFrame placeholder'ları oluşturuldu; sunum hücresi kırılmayacak.")
else:
    # ── Sözlük kopyası ve derleme ────────────────────────────────────────────
    work_df = df_ok.copy()
    regex_col = "Regex/Patern" if "Regex/Patern" in work_df.columns else ("Regex" if "Regex" in work_df.columns else None)
    if regex_col is None:
        raise RuntimeError("Sözlükte 'Regex/Patern' veya 'Regex' sütunu yok.")

    work_df["Regex_Safe_Pat"]        = work_df.apply(_safe_regex_from_row, axis=1)
    work_df["CompiledRegex_Safe"]    = work_df["Regex_Safe_Pat"].apply(_compile_local)
    work_df["CompiledRegex_Fallback"]= work_df[regex_col].astype(str).apply(_compile_local)

    safe_ok = float(work_df["CompiledRegex_Safe"].notna().mean())*100
    fb_ok   = float(work_df["CompiledRegex_Fallback"].notna().mean())*100
    print(f"Safe derleme oranı: {safe_ok:.2f}% | Fallback derleme oranı: {fb_ok:.2f}%")

    # ── SAFE→FALLBACK tarayıcı ───────────────────────────────────────────────
    def _try_row_patterns(text: str, row) -> List[Tuple[int,int,str,str]]:
        out = []
        rx1 = row.get("CompiledRegex_Safe", None)
        rx2 = row.get("CompiledRegex_Fallback", None)
        if rx1 is not None:
            for m in _iter_finditer(rx1, text):
                out.append((m.start(), m.end(), m.group(0), "safe"))
        if not out and rx2 is not None:
            for m in _iter_finditer(rx2, text):
                out.append((m.start(), m.end(), m.group(0), "fallback"))
        return out

    _diag_rows: List[Dict[str, Any]] = []

    # ── Cümle skorlama (df_hits — 2-C + tür öncelikli 3-B) ───────────────────
    def score_sentence(sent_id: int, text: str, dict_df: pd.DataFrame) -> List[Dict[str,Any]]:
        tlow = _tr_lower_norm(text)
        rough_hits: List[Dict[str,Any]] = []
        anchor_pass, regex_tried = 0, 0

        for _, row in dict_df.iterrows():
            # Basit anchor filtresi (opsiyonel)
            anchor_ok = True
            if not BYPASS_ANCHOR_PREFILTER:
                base = (row.get("Kavram") or row.get("Terim") or "").strip()
                anchor = _tr_lower_norm(base.split()[0]) if base else ""
                if anchor and len(anchor) >= MIN_ANCHOR_LEN:
                    anchor_ok = (anchor in tlow)
            if not anchor_ok:
                continue
            anchor_pass += 1

            matches = _try_row_patterns(text, row)  # SAFE→FALLBACK
            if matches:
                regex_tried += 1
                is_policy = _policy_bucket_local(row.get("Politika","")) in {"daraltıcı","genişletici"}
                base = _row_polarity(row)  # politika ise 0.0, aksi ±1/0
                for s, e, mg, src in matches:
                    s0 = max(0, s - CONTEXT_CHARS)
                    e0 = min(len(text), e + CONTEXT_CHARS)
                    rough_hits.append({
                        "Cümle_ID": sent_id,
                        "start": s, "end": e,
                        "Eşleşme": mg,
                        "Bağlam": text[s0:e0],
                        "Terim": row.get("Terim",""),
                        "Tür": row.get("Tür",""),
                        "Kavram": row.get("Kavram",""),
                        "Politika": row.get("Politika",""),
                        "Regex": row.get(regex_col,""),
                        "Polarite_Base": base,
                        "Polarite_Final": base,
                        "Regex_Kaynağı": src,
                        # öncelik için
                        "IsPolicyTone": bool(is_policy),
                        "AbsPol": float(abs(base)),
                    })

        _diag_rows.append({
            "Cümle_ID": sent_id,
            "Anchor_Pass": anchor_pass,
            "Regex_Tried": regex_tried,
            "Raw_Matches": len(rough_hits),
            "Cümle_Önizleme": text[:120].replace("\n"," "),
        })

        if not rough_hits:
            return []

        keep_idx = set(_resolve_overlaps_priority(rough_hits))
        hits_no_overlap = [h for i, h in enumerate(rough_hits) if i in keep_idx]
        return _dedup_term_regex(hits_no_overlap)  # 2-C

    # ── df_hits (tekilleştirilmiş + tür öncelikli) ───────────────────────────
    all_hits: List[Dict[str,Any]] = []
    for _, r in df_clean.iterrows():
        sid = int(r["Cümle_ID"]); sent_text = r["Cümle_Orijinal"]
        all_hits.extend(score_sentence(sid, sent_text, work_df))

    diag_df = pd.DataFrame(_diag_rows)

    df_hits = (pd.DataFrame(all_hits)
               .sort_values(["Cümle_ID","start","end"])
               .reset_index(drop=True)) if all_hits else pd.DataFrame(columns=expected_cols)

    print(f"🔎 Tutulan eşleşme adedi (tür öncelikli overlap & 2-C sonrası): {len(df_hits)}")

    # ── df_hits_full (HAM SAYAÇ: overlap çözümü YOK, tekilleştirme YOK) ─────
    def collect_hits_full(sent_id: int, text: str, dict_df: pd.DataFrame) -> List[Dict[str,Any]]:
        raw = []
        for _, row in dict_df.iterrows():
            is_policy = _policy_bucket_local(row.get("Politika","")) in {"daraltıcı","genişletici"}
            base = _row_polarity(row)
            for s, e, mg, src in _try_row_patterns(text, row):
                s0 = max(0, s - CONTEXT_CHARS)
                e0 = min(len(text), e + CONTEXT_CHARS)
                raw.append({
                    "Cümle_ID": sent_id, "start": s, "end": e,
                    "Eşleşme": mg, "Bağlam": text[s0:e0],
                    "Terim": row.get("Terim",""), "Tür": row.get("Tür",""),
                    "Kavram": row.get("Kavram",""), "Politika": row.get("Politika",""),
                    "Regex": row.get(regex_col,""),
                    "Polarite_Base": base, "Polarite_Final": base,
                    "Regex_Kaynağı": src,
                    "IsPolicyTone": bool(is_policy),
                    "AbsPol": float(abs(base)),
                })
        return raw

    all_hits_full = []
    for _, r in df_clean.iterrows():
        sid = int(r["Cümle_ID"]); sent_text = r["Cümle_Orijinal"]
        all_hits_full.extend(collect_hits_full(sid, sent_text, work_df))

    df_hits_full = (pd.DataFrame(all_hits_full)
                    .sort_values(["Cümle_ID","start","end"])
                    .reset_index(drop=True)) if all_hits_full else pd.DataFrame(columns=expected_cols)

    print(f"📦 df_hits_full oluşturuldu (HAM sayaç; overlap çözümü YOK, tekilleştirme YOK). Satır sayısı: {len(df_hits_full)}")

    # ── Cümle skoru (4-B) — politika tonu skordan ve paydadan tamamen hariç ──
    if df_hits.empty:
        # boş ise varsayılan skor yapıları
        sent_scores = df_clean[["Cümle_ID"]].copy()
        sent_scores["Eşleşme_Sayısı"]      = 0
        sent_scores["Eşleşme_Sayısı_Skor"] = 0
        sent_scores["Cümle_Skor_RawSum"]   = 0.0
        sent_scores["Cümle_Skor"]          = 0.0
        df_hits["Score_Include"]           = pd.Series(dtype=bool)
    else:
        df_hits["Score_Include"] = True
        if POLICY_TONE_NEUTRALIZE and "Politika" in df_hits.columns:
            _pt_mask_hits = df_hits["Politika"].apply(lambda v: _policy_bucket_local(v) in {"daraltıcı","genişletici"})
            df_hits.loc[_pt_mask_hits, "Score_Include"] = False

        def _normalize_sum_n(sum_val: float, n: int) -> float:
            if n <= 0: return 0.0
            return float(sum_val) / (1.0 + math.log(n))

        g = df_hits.groupby("Cümle_ID", dropna=False)
        all_cnt = g.size().rename("Eşleşme_Sayısı")
        sum_incl = g.apply(lambda x: x.loc[x["Score_Include"], "Polarite_Final"].sum()).rename("Cümle_Skor_RawSum")
        n_incl   = g["Score_Include"].sum().rename("Eşleşme_Sayısı_Skor")

        norm = pd.DataFrame({
            "Cümle_ID": sum_incl.index,
            "Cümle_Skor": [
                _normalize_sum_n(s, int(n))
                for s, n in zip(sum_incl.values, n_incl.values)
            ],
        }).set_index("Cümle_ID")

        sent_scores = pd.concat([all_cnt, n_incl, sum_incl, norm], axis=1).reset_index()

    df_sentence_scores = (df_clean[["Cümle_ID","Cümle_Orijinal","Cümle_Temiz"]]
                          .merge(sent_scores, on="Cümle_ID", how="left")
                          .fillna({"Eşleşme_Sayısı":0,
                                   "Eşleşme_Sayısı_Skor":0,
                                   "Cümle_Skor_RawSum":0.0,
                                   "Cümle_Skor":0.0}))

    # ============== 4-Z) Özel Analitik Dönüşüm (CUSTOM_TRANSFORM) ==============
    def _safe_call_custom_transform():
        if 'CUSTOM_TRANSFORM' not in globals():
            return None
        state_in = {
            'df_ok': df_ok,
            'df_clean': df_clean,
            'df_hits': df_hits,
            'df_hits_full': df_hits_full,
            'df_sentence_scores': df_sentence_scores,
            'ts_summary_df': ts_summary_df
        }
        res = CUSTOM_TRANSFORM(state_in)
        # Beklenen: df_hits, df_hits_full, df_sentence_scores, ts_summary_df, extra_state (opsiyonel)
        if isinstance(res, tuple) and len(res) >= 4:
            return res
        return None

    _try = _safe_call_custom_transform()
    if _try is not None:
        df_hits, df_hits_full, df_sentence_scores, ts_summary_df = _try[:4]
        CUSTOM_STATE = _try[4] if len(_try) > 4 else {}
        print("🔧 CUSTOM_TRANSFORM uygulandı.")
    else:
        CUSTOM_STATE = {}

    # ── Doküman düzeyi oranlar (TSOI tipi; ham sayaçtan) ────────────────────
    pos_doc = int((df_hits_full["Polarite_Final"] > 0).sum()) if not df_hits_full.empty else 0
    neg_doc = int((df_hits_full["Polarite_Final"] < 0).sum()) if not df_hits_full.empty else 0
    pn_total = pos_doc + neg_doc

    tokens_total = int(
        df_sentence_scores["Cümle_Temiz"].fillna("").astype(str).str.split().map(len).sum()
    ) if len(df_sentence_scores) else 0

    ts_net       = (pos_doc - neg_doc) / pn_total if pn_total > 0 else 0.0
    ts_posshare  =  pos_doc / pn_total            if pn_total > 0 else 0.0
    ts_net_per1k = 1000.0 * (pos_doc - neg_doc) / max(1, tokens_total)

    ts_summary_df = pd.DataFrame([{
        "pos": pos_doc, "neg": neg_doc,
        "Net Oran": round(ts_net, 4),
        "Pozitif Pay": round(ts_posshare, 4),
        "Net/1000 kelime": round(ts_net_per1k, 4),
        "Toplam kelime": tokens_total
    }])

    # ── Teşhis ve kontrol çıktıları ─────────────────────────────────────────
    def _ngram_counts(df):
        return {} if df is None or df.empty or "Tür" not in df.columns else df["Tür"].value_counts().to_dict()

    print("\n📚 N-gram dağılımı (df_hits):", _ngram_counts(df_hits))
    print("📚 N-gram dağılımı (df_hits_full/ham):", _ngram_counts(df_hits_full))

    doc_sum   = float(df_sentence_scores["Cümle_Skor"].sum()) if len(df_sentence_scores) else 0.0
    doc_mean  = float(df_sentence_scores["Cümle_Skor"].mean()) if len(df_sentence_scores) else 0.0
    doc_med   = float(df_sentence_scores["Cümle_Skor"].median()) if len(df_sentence_scores) else 0.0
    doc_var   = float(df_sentence_scores["Cümle_Skor"].var(ddof=1)) if len(df_sentence_scores) > 1 else 0.0

    print("\n📈 Doküman Özeti (4-B normalizasyonlu cümle skorları; df_hits)")
    print(f" • Toplam Skor   : {round(doc_sum, 4)}")
    print(f" • Ortalama Skor : {round(doc_mean, 4)}")
    print(f" • Medyan        : {round(doc_med, 4)}")
    print(f" • Varyans       : {round(doc_var, 6)}")
    print(f" • Toplam Cümle  : {len(df_sentence_scores)}")

    print("\n📊 TSOI-Tipi Oran Göstergeleri (df_hits_full H A M)")
    print(f" • Pozitif (pos) : {pos_doc}")
    print(f" • Negatif (neg) : {neg_doc}")
    print(f" • Net Oran      : {ts_net:.4f}")
    print(f" • Pozitif Pay   : {ts_posshare:.4f}")
    print(f" • Net/1000 kelime: {ts_net_per1k:.4f}")

    # Politika tonu skor katkısı (beklenen 0.0)
    if not df_hits_full.empty and "Politika" in df_hits_full.columns:
        _pol_mask_full = df_hits_full["Politika"].apply(lambda v: _policy_bucket_local(v) in {"daraltıcı","genişletici"})
        _pol_contrib = float(df_hits_full.loc[_pol_mask_full, "Polarite_Final"].sum()) if _pol_mask_full.any() else 0.0
        print(f"\n🛡️ Politika tonu skor katkısı (beklenen 0.0): {round(_pol_contrib, 4)}")

    # Normalizasyona dahil edilen / hariç bırakılan hit sayısı
    if not df_hits.empty and "Score_Include" in df_hits.columns:
        excl = int((~df_hits["Score_Include"]).sum())
        incl = int(df_hits["Score_Include"].sum())
        print(f"🧮 Normalizasyona dahil edilen hit sayısı: {incl} | Hariç tutulan (politika tonu): {excl}")

    # Politika tonu — n-gram & token sayımı (bilgi)
    def _norm_ngram_local(v: str) -> str:
        t = (v or "").strip().lower()
        if t.startswith("tri"): return "trigram"
        if t.startswith("bi"):  return "bigram"
        if t.startswith("uni"): return "unigram"
        return t or "unigram"

    def _tok_count(s: str) -> int:
        return len(RE.findall(r"[A-Za-zÇĞİÖŞÜçğıöşü]+", str(s or "")))

    policy_tone_hits = pd.DataFrame([])
    if not df_hits_full.empty and "Politika" in df_hits_full.columns:
        _tmp = df_hits_full.copy()
        _tmp["Politika_norm"] = _tmp["Politika"].apply(_policy_bucket_local)
        _tmp["Tür_norm"] = _tmp["Tür"].apply(_norm_ngram_local) if "Tür" in _tmp.columns else "unigram"
        _mask = _tmp["Politika_norm"].isin({"daraltıcı","genişletici"})

        if _mask.any():
            _tmp["Match_Token"] = _tmp["Eşleşme"].apply(_tok_count)
            hits_cnt = (_tmp.loc[_mask]
                        .groupby(["Politika_norm","Tür_norm"])
                        .size().reset_index(name="Ngram_Hit_Sayısı"))
            tok_cnt  = (_tmp.loc[_mask]
                        .groupby(["Politika_norm"])
                        ["Match_Token"].sum().reset_index(name="Eşleşme_Token_Toplamı"))
            policy_tone_hits = hits_cnt.merge(tok_cnt, on="Politika_norm", how="left").sort_values(
                ["Politika_norm","Tür_norm"]
            )
    if not policy_tone_hits.empty:
        print("\n🧭 Politika tonu — n-gram ve token sayımı (skordan bağımsız)")
        display(policy_tone_hits)


Safe derleme oranı: 95.24% | Fallback derleme oranı: 100.00%
🔎 Tutulan eşleşme adedi (tür öncelikli overlap & 2-C sonrası): 16
📦 df_hits_full oluşturuldu (HAM sayaç; overlap çözümü YOK, tekilleştirme YOK). Satır sayısı: 26

📚 N-gram dağılımı (df_hits): {'unigram': 7, 'bigram': 6, 'trigram': 3}
📚 N-gram dağılımı (df_hits_full/ham): {'unigram': 14, 'bigram': 8, 'trigram': 4}

📈 Doküman Özeti (4-B normalizasyonlu cümle skorları; df_hits)
 • Toplam Skor   : -0.5436
 • Ortalama Skor : -0.0453
 • Medyan        : 0.0
 • Varyans       : 0.912416
 • Toplam Cümle  : 12

📊 TSOI-Tipi Oran Göstergeleri (df_hits_full H A M)
 • Pozitif (pos) : 11
 • Negatif (neg) : 13
 • Net Oran      : -0.0833
 • Pozitif Pay   : 0.4583
 • Net/1000 kelime: -10.6383

🛡️ Politika tonu skor katkısı (beklenen 0.0): 0.0
🧮 Normalizasyona dahil edilen hit sayısı: 15 | Hariç tutulan (politika tonu): 1

🧭 Politika tonu — n-gram ve token sayımı (skordan bağımsız)


/tmp/ipython-input-2728702750.py:402: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sum_incl = g.apply(lambda x: x.loc[x["Score_Include"], "Polarite_Final"].sum()).rename("Cümle_Skor_RawSum")


,Politika_norm,Tür_norm,Ngram_Hit_Sayısı,Eşleşme_Token_Toplamı
0,daraltıcı,trigram,2,13


In [6]:
# @title
# ── Politika Duruşu (cümle bazlı, n-gram öncelikli) v3.5 + TEŞHİS ───────────
# Girdi : df_hits_full (2-A)  ve df_clean (Cümle_ID, Cümle_Orijinal)
# Çıktı : df_policy_sent, df_policy_counts  ve daraltıcı cümlelerin listesi

import pandas as pd, re, unicodedata
from IPython.display import display, HTML
import html

def _norm_tr(s):
    if s is None: return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("İ","i").replace("I","ı").lower()
    return unicodedata.normalize("NFC", s).strip()

# kökler
DAR_ROOTS = ("sıkı", "daralt", "sıkılaştır")
GEN_ROOTS = ("geniş", "gevş")

def _contains_any(t, roots):
    tt = _norm_tr(t);
    return any(r in tt for r in roots)

# özel trigram: sık* … para … politik*
def _has_siki_para_politika(t: str) -> bool:
    toks = [w for w in re.split(r"\s+", _norm_tr(t)) if w]
    if not toks: return False
    idx_s = [i for i,w in enumerate(toks) if w.startswith("sık")]
    idx_p = [i for i,w in enumerate(toks) if w.startswith("para")]
    idx_k = [i for i,w in enumerate(toks) if w.startswith("politik")]
    for i in idx_s:
        for j in idx_p:
            for k in idx_k:
                if i < j < k and (j-i) <= 3 and (k-j) <= 3:
                    return True
    return False

def _classify_text(txt: str) -> str:
    if not txt: return ""
    if _has_siki_para_politika(txt) or _contains_any(txt, DAR_ROOTS):
        return "daraltıcı"
    if _contains_any(txt, GEN_ROOTS):
        return "genişletici"
    return ""

def _norm_ngram(v: str) -> str:
    t = _norm_tr(v)
    if t in ("trigram","tri","3gram","3-gram"): return "trigram"
    if t in ("bigram","bi","2gram","2-gram"):  return "bigram"
    if t in ("unigram","uni","1gram","1-gram"):return "unigram"
    return ""

def _first_existing(df, names):
    for n in names:
        if n in df.columns: return n
    return None

if 'df_hits_full' not in globals() or not isinstance(df_hits_full, pd.DataFrame) or df_hits_full.empty:
    print("⚠️ df_hits_full yok/boş.")
    df_policy_sent   = pd.DataFrame(columns=["Cümle_ID","Politika_Kategori_Sentence","Ngram_Level"])
    df_policy_counts = pd.DataFrame({"Politika_Kategori":["daraltıcı","genişletici"],"Adet":[0,0]})
else:
    src = df_hits_full.copy()

    # dinamik kolonlar
    COL_POL   = _first_existing(src, ["Politika"])
    COL_TERM  = _first_existing(src, ["Terim"])
    COL_MATCH = _first_existing(src, ["Eşleşme","Eslesme"])
    COL_REGEX = _first_existing(src, ["Regex","Regex/Patern"])
    COL_TYPE  = _first_existing(src, ["Tür","Tur","Type"])
    COL_CTX   = _first_existing(src, ["Bağlam","Baglam","Context"])

    # 1) satır sınıflaması (Politika → Terim → Eşleşme → Regex → Bağlam)
    def _row_label(row):
        for col in [COL_POL, COL_TERM, COL_MATCH, COL_REGEX]:
            if col:
                lab = _classify_text(row.get(col, ""))
                if lab:
                    return lab, col
        if COL_CTX:
            lab = _classify_text(row.get(COL_CTX, ""))
            if lab:
                return lab, "Bağlam"
        return "", ""

    src[["Politika_Kategori","Politika_Kaynak"]] = src.apply(
        lambda r: pd.Series(_row_label(r)), axis=1
    )
    raw_hits = src["Politika_Kategori"].isin(["daraltıcı","genişletici"]).sum()
    print(f"🔍 Satır bazında sinyal bulunan kayıt sayısı: {raw_hits} / {len(src)}")

    # 2) Tür_norm: varsa normalize et, yoksa tahmin et
    if COL_TYPE:
        src["Tür_norm"] = src[COL_TYPE].apply(_norm_ngram)
    else:
        src["Tür_norm"] = ""

    need_guess = src["Tür_norm"].eq("")
    if need_guess.any():
        src.loc[need_guess, "Tür_norm"] = "unigram"  # güvenli varsayılan

    # Bağlamdan gelen ve trigram paterni taşıyanları zorla TRIGRAM yap (öncelik için kritik)
    if COL_CTX:
        mask_ctx_tri = (src["Politika_Kaynak"]=="Bağlam") & src[COL_CTX].apply(_has_siki_para_politika)
        src.loc[mask_ctx_tri, "Tür_norm"] = "trigram"

    # yalnız D/G satırlar
    lab_src = src[src["Politika_Kategori"].isin(["daraltıcı","genişletici"])].copy()
    if lab_src.empty:
        print("⚠️ D/G sinyali yok.")
        df_policy_sent   = pd.DataFrame(columns=["Cümle_ID","Politika_Kategori_Sentence","Ngram_Level"])
        df_policy_counts = pd.DataFrame({"Politika_Kategori":["daraltıcı","genişletici"],"Adet":[0,0]})
    else:
        # 3) cümle bazında tek karar (öncelik: tri > bi > uni)
        prio = {"trigram":0, "bigram":1, "unigram":2}
        lab_src["ng_prio"] = lab_src["Tür_norm"].map(prio).fillna(99).astype(int)

        def _pick_sentence(grp: pd.DataFrame):
            m = grp.loc[grp["ng_prio"].eq(grp["ng_prio"].min())]
            labs = set(m["Politika_Kategori"].unique())
            if len(labs) == 1:
                level = m["Tür_norm"].iloc[0]
                return pd.Series({"Politika_Kategori_Sentence": labs.pop(), "Ngram_Level": level})
            return None  # aynı seviyede D/G çatışması: dışla

        rows = []
        for sid, g in lab_src.groupby("Cümle_ID"):
            res = _pick_sentence(g)
            if res is not None:
                rows.append({"Cümle_ID": sid,
                             "Politika_Kategori_Sentence": res["Politika_Kategori_Sentence"],
                             "Ngram_Level": res["Ngram_Level"]})
        df_policy_sent = pd.DataFrame(rows, columns=["Cümle_ID","Politika_Kategori_Sentence","Ngram_Level"])

        # 4) sayım (belirsiz yok)
        counts = (df_policy_sent["Politika_Kategori_Sentence"]
                  .value_counts()
                  .reindex(["daraltıcı","genişletici"], fill_value=0))
        df_policy_counts = counts.rename_axis("Politika_Kategori").reset_index(name="Adet")

# ----- ÇIKTI: sayım + daraltıcı cümleleri göster --------------------------------
_d = int(df_policy_counts.loc[df_policy_counts["Politika_Kategori"]=="daraltıcı","Adet"].sum()) if not df_policy_counts.empty else 0
_g = int(df_policy_counts.loc[df_policy_counts["Politika_Kategori"]=="genişletici","Adet"].sum()) if not df_policy_counts.empty else 0
print(f"🧭 Politika duruşu (cümle bazlı, n-gram öncelikli) — daraltıcı:{_d} | genişletici:{_g}")
display(df_policy_counts)

# Hangi cümlelerden? — df_clean ile birleştir ve vurgula
def _hl_siki_para(text: str) -> str:
    if not isinstance(text, str): return ""
    esc = html.escape(text)
    # sade vurgulama (case-insensitive)
    return re.sub(r'(?i)(sık[ıi]\s*para\s*politik\w*)',
                  r'<mark style="background:#fde68a;color:#000;border-radius:4px;padding:0 2px;">\1</mark>',
                  esc)

if 'df_clean' in globals() and not df_policy_sent.empty:
    dbg = (df_policy_sent[df_policy_sent["Politika_Kategori_Sentence"]=="daraltıcı"]
           .merge(df_clean[["Cümle_ID","Cümle_Orijinal"]], on="Cümle_ID", how="left")
           .sort_values("Cümle_ID"))
    if not dbg.empty:
        print("\n🔎 Daraltıcı bulunan cümleler:")
        display(dbg[["Cümle_ID","Ngram_Level"]])
        # HTML vurgulu metin
        html_rows = []
        for _, r in dbg.iterrows():
            html_rows.append(f"<div><b>#{int(r['Cümle_ID'])}</b> — {_hl_siki_para(r['Cümle_Orijinal'])}</div>")
        display(HTML("<div style='line-height:1.6;margin-top:6px;'>" + "<br>".join(html_rows) + "</div>"))


🔍 Satır bazında sinyal bulunan kayıt sayısı: 7 / 26
🧭 Politika duruşu (cümle bazlı, n-gram öncelikli) — daraltıcı:2 | genişletici:0


,Politika_Kategori,Adet
0,daraltıcı,2
1,genişletici,0



🔎 Daraltıcı bulunan cümleler:


,Cümle_ID,Ngram_Level
0,8,trigram
1,12,trigram


In [7]:
# @title
# ===============================================
# 📄🧭 COLAB HÜCRE 5 — Rapor & Dashboard (REVIZE v3.3)
# df_hits_full = HAM sayaç (diagnostic/coverage)
# df_hits      = Tür öncelikli, tekilleştirilmiş, skorun kaynağı
# Renkler:
#   Skora dahil:  koyu yeşil/kırmızı
#   Skora dahil değil (2-C’de tutulan): açık yeşil/kırmızı
#   Yalnız politika tonu: beyaz (ince kenarlık)
# ===============================================

import pandas as pd, numpy as np, math, html, unicodedata
import re as RE
from IPython.display import HTML, display
from io import StringIO
from collections import defaultdict

# ── Opsiyonel dışa aktarma (CSV)
EXPORT_CSV = False  # True yapılırsa CSV çıktıları üretir

# ── Ön koşullar
_needed = ["df_ok","df_hits","df_hits_full","df_sentence_scores","ts_summary_df","diag_df"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Bu hücreyi çalıştırmak için önce Hücre-4 tamamlanmalı. Eksik: {_missing}")

# ============== Yardımcılar ==============
def _html_escape(x: str) -> str:
    return html.escape("" if x is None else str(x), quote=False)

def _to_percent(x, digits=1):
    try:
        return f"{100*float(x):.{digits}f}%"
    except Exception:
        return "0%"

def _norm_tr(s):
    if s is None: return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("İ","i").replace("I","ı").lower()
    return unicodedata.normalize("NFC", s).strip()

def _norm_soft(s):
    s = _norm_tr(s)
    return RE.sub(r"\s+", " ", s)

def _as_bool(x):
    if isinstance(x, bool): return x
    if x is None: return False
    if isinstance(x, (int, float)): return bool(x)
    s = str(x).strip().lower()
    return s in ("1","true","yes","y","evet","doğru","included","include","on")

def _html_table(df: pd.DataFrame, escape=True, index=False, classes="compact"):
    return df.to_html(escape=escape, index=index, classes=classes)

def _get_token_source_col(df):
    if "Cümle_Temiz" in df.columns: return "Cümle_Temiz"
    if "Cümle_Orijinal" in df.columns: return "Cümle_Orijinal"
    return None

# STRICT: yalnızca sözlük etiketine göre (heuristik yok)
def _policy_bucket_strict(v: str) -> str:
    t = _norm_tr(v)
    if t == "daraltıcı":    return "daraltıcı"
    if t == "genişletici":  return "genişletici"
    return "belirsiz/boş"

# --- CSS (ışık/karanlık uyumlu) ---

style_css = """
<style>
  mark { background: transparent; color: inherit; }
  .sec-title { font-weight:700; font-size:16px; margin: 14px 0 6px 0; }
  .grid { display:grid; grid-template-columns: repeat(6, minmax(120px,1fr)); gap:12px; margin:12px 0; }
  .card { background:#0b1324; color:#e6edf3; border:1px solid #223; border-radius:14px; padding:12px; box-shadow:0 2px 8px rgba(0,0,0,.18); }
  .card .title { font-size:12px; opacity:.8; }
  .card .value { font-size:22px; font-weight:700; margin-top:4px; }
  .card .sub   { font-size:11px; opacity:.7; margin-top:2px; }
  @media (prefers-color-scheme: light) {
    .card { background:#f8fafc; color:#0f172a; border-color:#e2e8f0; }
  }
  mark.pos1 { background:#22c55e; color:#052e16; padding:0 2px; border-radius:4px; }
  mark.neg1 { background:#ef4444; color:#7f1d1d; padding:0 2px; border-radius:4px; }
  mark.pos2 { background:#bbf7d0; color:#065f46; padding:0 2px; border-radius:4px; }
  mark.neg2 { background:#fecaca; color:#7f1d1d; padding:0 2px; border-radius:4px; }
  mark.policy { background:#ffffff; color:#0f172a; padding:0 2px; border-radius:4px; border:1px solid #cbd5e1; }
  mark.neu  { background:#e5e7eb; color:#111827; padding:0 2px; border-radius:4px; }
  table.compact td, table.compact th { padding:6px 8px; font-size:13px; }
  .table-explain { font-size:12px; opacity:.8; margin:4px 0 8px 0; }
</style>
"""
display(HTML(style_css))

# ============== 0.5) Rapor çıktısı için sütun isimleri ve tablo açıklamaları ==============

# Cümle skoru tablosu
COL_MAP_SENT = {
    "Cümle_ID": "Cümle No",
    "Cümle_Orijinal": "Orijinal Cümle",
    "Cümle_Temiz": "Ön İşlemden Geçmiş Cümle",
    "Cümle_Skor_RawSum": "Ham Cümle Toplam Skoru",
    "Eşleşme_Sayısı": "Toplam Eşleşme Sayısı",
    "Eşleşme_Sayısı_Skor": "Skora Dahil Eşleşme Sayısı",
    "Cümle_Skor": "Normalize Cümle Skoru",
}

# Eşleşme tabloları (df_hits, df_hits_full)
COL_MAP_HITS = {
    "Cümle_ID": "Cümle No",
    "start": "Başlangıç Konumu (Karakter)",
    "end": "Bitiş Konumu (Karakter)",
    "Eşleşme": "Metinde Yakalanan İfade",
    "Bağlam": "Cümle Bağlamı",
    "Terim": "Sözlük Terimi",
    "Tür": "N-gram Türü",
    "Kavram": "Kavram Etiketi",
    "Politika": "Politika Tonu (Daraltıcı/Genişletici)",
    "Regex": "Kullanılan Regex Deseni",
    "Regex_Kaynağı": "Regex Kaynağı (SAFE/FALLBACK)",
    "Polarite_Base": "Ham Polarite Puanı",
    "Polarite_Final": "Düzeltilmiş Polarite Puanı",
    "Score_Include": "Skor Hesabına Dahil mi?",
}

# N-gram dağılımı
COL_MAP_NGRAM = {
    "Tür": "N-gram Türü",
    "Adet": "Gözlenen Frekans",
}

# Terim özetleri
COL_MAP_TERMS = {
    "Terim": "Sözlük Terimi",
    "Adet": "Metinde Görülme Sayısı",
    "Tür": "N-gram Türü",  # Tür × Terim tablosu için de kullanılacak
}

# Regex teşhis tablosu (diag_df)
COL_MAP_DIAG = {
    "Cümle_ID": "Cümle No",
    "Anchor_Pass": "Anchor Filtresi (Geçti/Kaldı)",
    "Regex_Tried": "Denenen Regex Sayısı",
    "Raw_Matches": "Ham Eşleşme Sayısı",
    "Cümle_Önizleme": "Cümle Önizleme",
}

# Tam metin vurgulu tablo (full_df)
COL_MAP_FULLTEXT = {
    "Cümle_ID": "Cümle No",
    "Cümle_Skor": "Normalize Cümle Skoru",
    "Eşleşme_Sayısı": "Toplam Eşleşme Sayısı",
    # "Metin (vurgulu)" zaten açıklayıcı, ismi koruyoruz
}

TABLE_DESCRIPTIONS = {
    "diag": (
        "<b>Regex Teşhis Tablosu:</b> Her cümle için kaç regex deseninin denendiğini ve "
        "kaç ham eşleşme bulunduğunu gösterir. Sözlüğün metinle ne kadar uyumlu çalıştığını "
        "kontrol etmek için kullanılır."
    ),
    "sentence": (
        "<b>Cümle Düzeyi Skor Tablosu:</b> Her cümlede sözlük terimleri üzerinden hesaplanan "
        "duygu/ton skorunu ve eşleşme sayılarını gösterir. Metnin genel tonunu bu skorlar üzerinden özetliyoruz."
    ),
    "hits": (
        "<b>Skora Dahil Eşleşmeler Tablosu:</b> Overlap çözümü ve tür önceliği sonrasında, "
        "cümle skorunun hesaplanmasında kullanılan tüm eşleşmeleri raporlar."
    ),
    "hits_full": (
        "<b>Ham Eşleşmeler Tablosu:</b> Sözlük ve regex desenleriyle metinde yakalanan tüm "
        "ifadelerin ham listesidir. Kapsama ve sözlük kalitesini değerlendirmek için kullanılır."
    ),
    "ngram_hits": (
        "<b>N-gram Dağılımı (Skora Dahil Eşleşmeler):</b> Skora dahil edilen eşleşmelerin "
        "unigram / bigram / trigram bazında frekans dağılımını gösterir."
    ),
    "ngram_full": (
        "<b>N-gram Dağılımı (Ham Eşleşmeler):</b> Ham düzeyde yakalanan tüm eşleşmelerin "
        "n-gram türlerine göre dağılımını gösterir."
    ),
    "terms_full": (
        "<b>Terim Sıklıkları (Ham Eşleşmeler):</b> Her sözlük teriminin metin boyunca kaç kez "
        "yakalandığını gösterir; sözlüğün metne ne kadar oturduğunu izlemeye yarar."
    ),
    "fulltext": (
        "<b>Tam Metin (vurgulu):</b> Her cümleyi orijinal sıralıyla gösterir; sözlük terimleri metin içinde "
        "renk kodlarıyla vurgulanmıştır. Cümle skorlarıyla birlikte okunabilir."
    ),
}

def _rename_for_report(df: pd.DataFrame, kind: str) -> pd.DataFrame:
    """
    df: Gösterilecek tablo
    kind: 'sentence', 'hits', 'hits_full', 'ngram', 'terms', 'diag', 'fulltext'
    """
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return df

    if kind == "sentence":
        cmap = COL_MAP_SENT
    elif kind in ("hits", "hits_full"):
        cmap = COL_MAP_HITS
    elif kind == "ngram":
        cmap = COL_MAP_NGRAM
    elif kind == "terms":
        cmap = COL_MAP_TERMS
    elif kind == "diag":
        cmap = COL_MAP_DIAG
    elif kind == "fulltext":
        cmap = COL_MAP_FULLTEXT
    else:
        cmap = {}

    cmap_effective = {k: v for k, v in cmap.items() if k in df.columns}
    if not cmap_effective:
        return df
    return df.rename(columns=cmap_effective)



# ============== Tür dönüşümleri (hijyen) ==============
def _coerce_types(df):
    df = df.copy()
    if "Score_Include" not in df.columns:
        df["Score_Include"] = False
    else:
        df["Score_Include"] = df["Score_Include"].map(_as_bool)
    if "Polarite_Final" not in df.columns:
        df["Polarite_Final"] = 0.0
    else:
        df["Polarite_Final"] = pd.to_numeric(df["Polarite_Final"], errors="coerce").fillna(0.0)
    for c in ["Politika","Terim","Eşleşme","Regex","Regex_Kaynağı","Tür","Kavram","Bağlam"]:
        if c in df.columns: df[c] = df[c].astype(str)
    for c in ["start","end","Cümle_ID"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

_df_hits      = _coerce_types(df_hits) if isinstance(df_hits, pd.DataFrame) else pd.DataFrame()
_df_hits_full = _coerce_types(df_hits_full) if isinstance(df_hits_full, pd.DataFrame) else pd.DataFrame()

# ============== 0) Kısa durum özeti ==============
regex_col = "Regex/Patern" if "Regex/Patern" in df_ok.columns else ("Regex" if "Regex" in df_ok.columns else None)
if regex_col is None:
    raise RuntimeError("Sözlükte 'Regex/Patern' veya 'Regex' sütunu yok.")
regex_fill_ratio = float(df_ok[regex_col].astype(str).str.strip().ne("").mean())*100.0
print(f"Regex sütunu: {regex_col} | Regex doluluk oranı: {regex_fill_ratio:.2f}%")

print("\n🧪 Teşhis Özeti (tüm cümleler):")
if isinstance(diag_df, pd.DataFrame) and not diag_df.empty:
    cols_diag = [
        c for c in ["Cümle_ID","Anchor_Pass","Regex_Tried","Raw_Matches","Cümle_Önizleme"]
        if c in diag_df.columns
    ]
    # TÜM SATIRLAR – .head(10) YOK
    _diag_view_raw = diag_df.loc[:, cols_diag]
    _diag_view = _rename_for_report(_diag_view_raw, kind="diag")
    desc = TABLE_DESCRIPTIONS.get("diag", "")
    if desc:
        display(HTML(f"<div class='table-explain'>{desc}</div>"))
    display(HTML(_html_table(_diag_view)))
else:
    print("— (boş) —")

# ============== 1) Üst özet kartları ==============
if "Cümle_Skor" not in df_sentence_scores.columns:
    df_sentence_scores = df_sentence_scores.copy()
    df_sentence_scores["Cümle_Skor"] = 0.0

doc_sum   = float(df_sentence_scores["Cümle_Skor"].sum()) if len(df_sentence_scores) else 0.0
doc_mean  = float(df_sentence_scores["Cümle_Skor"].mean()) if len(df_sentence_scores) else 0.0
doc_med   = float(df_sentence_scores["Cümle_Skor"].median()) if len(df_sentence_scores)>0 else 0.0
doc_var   = float(df_sentence_scores["Cümle_Skor"].var(ddof=1)) if len(df_sentence_scores)>1 else 0.0

pos_doc = int((_df_hits_full["Polarite_Final"] > 0).sum()) if not _df_hits_full.empty else 0
neg_doc = int((_df_hits_full["Polarite_Final"] < 0).sum()) if not _df_hits_full.empty else 0
pn_total = pos_doc + neg_doc

tok_col = _get_token_source_col(df_sentence_scores)
tokens_total = int(
    df_sentence_scores[tok_col].fillna("").astype(str).str.split().map(len).sum()
) if len(df_sentence_scores) and tok_col else 0

ts_net       = (pos_doc - neg_doc) / pn_total if pn_total > 0 else 0.0
ts_posshare  =  pos_doc / pn_total            if pn_total > 0 else 0.0
ts_net_per1k = 1000.0 * (pos_doc - neg_doc) / max(1, tokens_total)
coverage     = float((df_sentence_scores["Eşleşme_Sayısı"]>0).mean()) if ("Eşleşme_Sayısı" in df_sentence_scores.columns and len(df_sentence_scores)) else 0.0

_ngram_kapsam_val = "0"
if not _df_hits.empty and ("Tür" in _df_hits.columns):
    _ngram_kapsam_val = str(_df_hits["Tür"].value_counts().to_dict()).replace("'", "")

cards_html = f"""
<div class="grid">
  <div class="card"><div class="title">Doküman Toplam Skor</div><div class="value">{doc_sum:.4f}</div><div class="sub">4-B normalizasyonlu (df_hits)</div></div>
  <div class="card"><div class="title">Ortalama Cümle Skoru</div><div class="value">{doc_mean:.4f}</div><div class="sub">Medyan {doc_med:.2f}</div></div>
  <div class="card"><div class="title">Skor Varyansı</div><div class="value">{doc_var:.6f}</div><div class="sub">Cümle skor dağılımı</div></div>
  <div class="card"><div class="title">Pozitif/Negatif (HAM)</div><div class="value">{pos_doc}/{neg_doc}</div><div class="sub">df_hits_full n={pn_total}</div></div>
  <div class="card"><div class="title">Net Oran (HAM)</div><div class="value">{ts_net:.4f}</div><div class="sub">Pozitif pay {_to_percent(ts_posshare)}</div></div>
  <div class="card"><div class="title">Net / 1000 kelime (HAM)</div><div class="value">{ts_net_per1k:.2f}</div><div class="sub">Toplam kelime {tokens_total}</div></div>
</div>
<div class="grid" style="grid-template-columns: repeat(3, minmax(120px,1fr));">
  <div class="card"><div class="title">Kapsama</div><div class="value">{_to_percent(coverage)}</div><div class="sub">≥1 eşleşme içeren cümle oranı (df_hits)</div></div>
  <div class="card"><div class="title">N-gram kapsamı</div><div class="value">{_ngram_kapsam_val}</div><div class="sub">df_hits (tekilleştirilmiş)</div></div>
</div>
<div class="sec-title">🎨 Renk Efsanesi</div>
<div class="card" style="grid-column: 1 / -1;">
  <div class="sub">
    Skora dahil: <mark class="pos1">pozitif</mark> / <mark class="neg1">negatif</mark> &nbsp; | &nbsp;
    Skora dahil değil (2-C): <mark class="pos2">pozitif</mark> / <mark class="neg2">negatif</mark> &nbsp; | &nbsp;
    Politika tonu: <mark class="policy">daraltıcı / genişletici</mark>
  </div>
</div>
"""
display(HTML(cards_html))

# ============== 1.1) CUSTOM_STATE → ekstra sunum bölümleri (opsiyonel) ==============
if 'CUSTOM_STATE' in globals() and isinstance(CUSTOM_STATE, dict):
    # HTML blok(lar)ı
    if isinstance(CUSTOM_STATE.get("DASHBOARD_HTML"), str) and CUSTOM_STATE["DASHBOARD_HTML"].strip():
        display(HTML('<div class="sec-title">🧩 Özel Sunum</div>'))
        display(HTML(CUSTOM_STATE["DASHBOARD_HTML"]))
    # Tablo(lar)
    if isinstance(CUSTOM_STATE.get("TABLES"), dict) and len(CUSTOM_STATE["TABLES"]):
        display(HTML('<div class="sec-title">🧩 Özel Tablolar</div>'))
        for name, df_ in CUSTOM_STATE["TABLES"].items():
            try:
                display(HTML(f"<div class='sec-title' style='font-size:14px'>{_html_escape(name)}</div>"))
                display(HTML(_html_table(df_ if isinstance(df_, pd.DataFrame) else pd.DataFrame(df_))))
            except Exception:
                pass

# ============== 2) Politika duruşu (STRICT — sözlük etiketi) ==============
if not _df_hits_full.empty and "Politika" in _df_hits_full.columns:
    df_policy_counts_strict = (
        _df_hits_full.assign(Politika_Kategori=lambda d: d["Politika"].map(_policy_bucket_strict))
                    .groupby("Politika_Kategori").size()
                    .reindex(["daraltıcı","genişletici","belirsiz/boş"], fill_value=0)
                    .reset_index(name="Adet")
    )
else:
    df_policy_counts_strict = pd.DataFrame({
        "Politika_Kategori": ["daraltıcı","genişletici","belirsiz/boş"],
        "Adet": [0,0,0]
    })

display(HTML('<div class="sec-title">🧭 Politika Duruşu Dağılımı (Sadece Sözlük Etiketi; HAM)</div>'))
display(HTML(_html_table(df_policy_counts_strict)))

# ============== 3) Vurgu sınıfı ve başlık (tooltip) üretimi ==============
def _span_class_from_row(r):
    p = float(r.get("Polarite_Final", 0) or 0)
    include = _as_bool(r.get("Score_Include", False))
    pol_lab = str(r.get("Politika", "") or "")
    pol_bucket = _policy_bucket_strict(pol_lab) if pol_lab else "belirsiz/boş"

    if include:
        if p > 0:  return "pos1"
        if p < 0:  return "neg1"
        return "neu"

    if p != 0:
        return "pos2" if p > 0 else "neg2"

    if pol_bucket in ("daraltıcı","genişletici"):
        return "policy"

    return "neu"

def _mk_title(r):
    p   = float(r.get("Polarite_Final", 0) or 0.0)
    inc = "dahil" if _as_bool(r.get("Score_Include", False)) else "hariç"
    ter = str(r.get("Terim","") or "")
    esl = str(r.get("Eşleşme","") or "")
    pol = str(r.get("Politika","") or "")
    rx  = str(r.get("Regex","") or "")
    src = str(r.get("Regex_Kaynağı","") or "")
    return f"Terim: {ter}\nEşleşme: {esl}\nPolarite: {p:+.3f}\nSkor: {inc}\nPolitika: {pol}\nRegex: {rx}\nKaynak: {src}"

def _build_span_maps(df_hits, df_hits_full):
    prim_by_sid, prim_cls_by_sid, prim_tip_by_sid = defaultdict(list), defaultdict(list), defaultdict(list)
    viz_keys = set()

    if df_hits is not None and not df_hits.empty:
        for _, r in df_hits.iterrows():
            sid = int(r["Cümle_ID"]); s,e = int(r["start"]), int(r["end"])
            prim_by_sid[sid].append((s,e))
            prim_cls_by_sid[sid].append(_span_class_from_row(r))
            prim_tip_by_sid[sid].append(_mk_title(r))
            viz_keys.add((sid,s,e,str(r.get("Terim","")),str(r.get("Regex",""))))

    extra_by_sid, extra_cls_by_sid, extra_tip_by_sid = defaultdict(list), defaultdict(list), defaultdict(list)
    if df_hits_full is not None and not df_hits_full.empty:
        for _, r in df_hits_full.iterrows():
            sid = int(r["Cümle_ID"]); s,e = int(r["start"]), int(r["end"])
            key = (sid,s,e,str(r.get("Terim","")),str(r.get("Regex","")))
            if key in viz_keys:
                continue
            extra_by_sid[sid].append((s,e))
            extra_cls_by_sid[sid].append(_span_class_from_row(r))
            extra_tip_by_sid[sid].append(_mk_title(r))

    return prim_by_sid, prim_cls_by_sid, prim_tip_by_sid, extra_by_sid, extra_cls_by_sid, extra_tip_by_sid

def _highlight_spans(text, prim_spans, prim_classes, prim_titles, extra_spans, extra_classes, extra_titles):
    if not text:
        return ""
    kept, occupied = [], []
    def _add_if_free(s, e, cls, tip):
        for s2, e2, _ in occupied:
            if not (e <= s2 or e2 <= s):
                return False
        kept.append((s, e, cls, tip)); occupied.append((s, e, cls)); return True

    if prim_spans:
        for (s, e), cls, tip in sorted(zip(prim_spans, prim_classes, prim_titles), key=lambda t: t[0][0]):
            _add_if_free(int(s), int(e), cls, tip)
    if extra_spans:
        for (s, e), cls, tip in sorted(zip(extra_spans, extra_classes, extra_titles), key=lambda t: t[0][0]):
            _add_if_free(int(s), int(e), cls, tip)

    kept.sort(key=lambda x: x[0])
    pieces, cursor = [], 0
    for s, e, cls, tip in kept:
        if s > cursor:
            pieces.append(_html_escape(text[cursor:s]))
        tip_attr = _html_escape(tip) if tip else ""
        pieces.append(f'<mark class="{cls}" title="{tip_attr}">{_html_escape(text[s:e])}</mark>')
        cursor = e
    pieces.append(_html_escape(text[cursor:]))
    return "".join(pieces)

# ============== 4) “Sıkı para politikası” — STRICT vs DEBUG sayaçları ==============
EXPECTED_SPM_COUNT = globals().get("EXPECTED_SPM_COUNT", None)  # örn. 2
if not _df_hits_full.empty:
    _tmp = _df_hits_full.copy()
    # Güvenli norm kolonları
    _tmp["Terim_norm"]   = _tmp["Terim"].astype(str).map(_norm_soft)   if "Terim"   in _tmp.columns else pd.Series("", index=_tmp.index)
    _tmp["Eşleşme_norm"] = _tmp["Eşleşme"].astype(str).map(_norm_soft) if "Eşleşme" in _tmp.columns else pd.Series("", index=_tmp.index)

    mask_terim   = (_tmp["Terim_norm"] == "sıkı para politikası")
    mask_eslesme = _tmp["Eşleşme_norm"].str.contains(r"\bsık[ıi]\s+para\s+politikası\b", regex=True, na=False)

    spm_full_strict = _tmp.loc[mask_terim].copy()
    spm_full_debug  = _tmp.loc[mask_eslesme].copy()

    full_count_strict = len(spm_full_strict)
    full_count_debug  = len(spm_full_debug)
    uniq_sent_strict  = spm_full_strict["Cümle_ID"].nunique() if "Cümle_ID" in spm_full_strict.columns else full_count_strict
    uniq_sent_debug   = spm_full_debug["Cümle_ID"].nunique()  if "Cümle_ID" in spm_full_debug.columns  else full_count_debug

    print(f"\n🧮 'sıkı para politikası' — STRICT(Terim): {full_count_strict} satır / {uniq_sent_strict} cümle | "
          f"DEBUG(Eşleşme): {full_count_debug} satır / {uniq_sent_debug} cümle")

    if "Politika" in spm_full_strict.columns and not spm_full_strict.empty:
        print("   Kovalar (STRICT/sözlük):", spm_full_strict["Politika"].map(_policy_bucket_strict).value_counts(dropna=False).to_dict())
else:
    print("\nℹ️ df_hits_full boş; 'sıkı para politikası' kontrolü yapılamadı.")

# ============== 5) Eşleşme tablosu (df_hits) ==============
print("\n🔎 Seçili Eşleşmeler Tablosu "
      "(Adım 3-B — Tür Öncelikli Çakışma Çözümü + "
      "Adım 2-C — Kapsama Amaçlı Eşleşme Filtresi sonrası)")
print(f"   Toplam eşleşme satırı: {len(_df_hits)}")

if isinstance(_df_hits, pd.DataFrame) and not _df_hits.empty:
    # Bu tablo: skor hesaplamasında kullanılan, tür öncelikli ve tekilleştirilmiş eşleşmeler
    cols_hits = [
        c for c in [
            "Cümle_ID",        # cümle kimliği
            "start", "end",    # metin içinde başlangıç / bitiş konumu
            "Eşleşme",         # metinden yakalanan ifade
            "Bağlam",          # eşleşmenin etrafındaki kısa bağlam
            "Terim",           # sözlükteki kavram etiketi
            "Tür",             # n-gram türü (unigram / bigram / trigram)
            "Kavram",          # üst kavram / tema
            "Politika",        # politika tonu etiketi (daraltıcı / genişletici / boş)
            "Regex",           # kullanılan regex paterni
            "Polarite_Base",   # sözlükteki ham polarite
            "Polarite_Final",  # bağlam/filtre sonrası nihai polarite
            "Regex_Kaynağı",   # paternin güvenlik/kaynak etiketi (safe / fallback vb.)
            "Score_Include"    # bu satır belge skoruna dahil mi? (True/False)
        ]
        if c in _df_hits.columns
    ]

    _view_hits_raw = _df_hits.loc[:, cols_hits].sort_values(["Cümle_ID", "start"])

    # Sütun adlarını tez okuyucusu için sadeleştir
    _view_hits = _rename_for_report(_view_hits_raw, kind="hits")

    # Tablo açıklamasını üstte göster (TABLE_DESCRIPTIONS['hits'] içinde tanımlı olmalı)
    desc = TABLE_DESCRIPTIONS.get("hits", "")
    if desc:
        display(HTML(f"<div class='table-explain'>{desc}</div>"))

    # Tabloyu göster
    display(HTML(_html_table(_view_hits)))
else:
    print("— Bu aşamada (3-B + 2-C sonrası) tutulmuş eşleşme bulunmamaktadır. —")


# ============== 6) N-gram dağılımları ==============
def _ngram_table(df):
    if df is None or df.empty or "Tür" not in df.columns: return None
    return (df["Tür"].value_counts()
            .rename_axis("Tür").reset_index(name="Adet")
            .sort_values(["Adet","Tür"], ascending=[False, True]))

ng_counts_hits = _ngram_table(_df_hits)
if ng_counts_hits is not None:
    display(HTML('<div class="sec-title">📚 N-gram Yakalama Dağılımı (df_hits)</div>'))
    _ngram_hits_view = _rename_for_report(ng_counts_hits, kind="ngram")
    desc = TABLE_DESCRIPTIONS.get("ngram_hits", "")
    if desc:
        display(HTML(f"<div class='table-explain'>{desc}</div>"))
    display(HTML(_html_table(_ngram_hits_view)))

# Tür × Terim
if isinstance(_df_hits, pd.DataFrame) and not _df_hits.empty and "Tür" in _df_hits.columns and "Terim" in _df_hits.columns:
    per_term = (_df_hits.groupby(["Tür","Terim"], dropna=False)
                .size().reset_index(name="Adet")
                .sort_values(["Tür","Adet","Terim"], ascending=[True, False, True]))
    display(HTML('<div class="sec-title">🏷️ Tür × Terim hit sayıları (azalan)</div>'))
    display(HTML(_html_table(per_term)))

# ============== 7) Cümle skor tablosu ==============
def _safe(val, fn, default=0.0):
    try:
        return float(fn(val))
    except Exception:
        return default

if isinstance(df_sentence_scores, pd.DataFrame) and not df_sentence_scores.empty:
    doc_sum  = _safe(df_sentence_scores["Cümle_Skor"].sum(), lambda x: x)
    doc_mean = _safe(df_sentence_scores["Cümle_Skor"].mean(), lambda x: x)
    doc_med  = _safe(df_sentence_scores["Cümle_Skor"].median(), lambda x: x)
    doc_var  = _safe(df_sentence_scores["Cümle_Skor"].var(ddof=1), lambda x: x)
    print("\n📈 Doküman Özeti (4-B normalizasyonlu cümle skorları; df_hits)")
    print(f" • Toplam Skor   : {doc_sum:.4f}")
    print(f" • Ortalama Skor : {doc_mean:.4f}")
    print(f" • Medyan        : {doc_med:.4f}")
    print(f" • Varyans       : {doc_var:.6f}")
    print(f" • Toplam Cümle  : {len(df_sentence_scores)}")

print("\n🧾 Cümle Skor Tablosu (tüm cümleler):")

cols_sent = [c for c in [
    "Cümle_ID",
    "Cümle_Orijinal",
    "Cümle_Temiz",
    "Cümle_Skor_RawSum",
    "Eşleşme_Sayısı",
    "Eşleşme_Sayısı_Skor",
    "Cümle_Skor"
] if c in df_sentence_scores.columns]

if cols_sent:
    # Tüm cümleleri, ID sırasına göre göster
    _view_sent_raw = df_sentence_scores.loc[:, cols_sent].sort_values("Cümle_ID")
else:
    # Her ihtimale karşı, kolon listesi boşsa ham dataframe’i göster
    _view_sent_raw = df_sentence_scores.copy()

# Tez okuyucusuna uygun sütun adları
_view_sent = _rename_for_report(_view_sent_raw, kind="sentence")

# Tablo açıklaması (TABLE_DESCRIPTIONS["sentence"] içinde tanımlıysa)
desc = TABLE_DESCRIPTIONS.get("sentence", "")
if desc:
    display(HTML(f"<div class='table-explain'>{desc}</div>"))

display(HTML(_html_table(_view_sent)))


# ============== 8) Vurgulanan ifadeler & dağılımlar (df_hits_full) ==============
if isinstance(_df_hits_full, pd.DataFrame) and not _df_hits_full.empty:
    cols_full = [c for c in ["Cümle_ID","start","end","Eşleşme","Terim","Tür","Politika","Polarite_Final","Score_Include"] if c in _df_hits_full.columns]
    if cols_full:
        display(HTML('<div class="sec-title">🔎 Yakalanan İfadeler (df_hits_full — HAM, ilk 50)</div>'))
        _view_full_raw = _df_hits_full.loc[:, cols_full].sort_values(["Cümle_ID","start"]).head(50)
        _view_full = _rename_for_report(_view_full_raw, kind="hits_full")
        desc = TABLE_DESCRIPTIONS.get("hits_full", "")
        if desc:
            display(HTML(f"<div class='table-explain'>{desc}</div>"))
        display(HTML(_html_table(_view_full)))


ng_counts_full = _ngram_table(_df_hits_full)
if ng_counts_full is not None:
    display(HTML('<div class="sec-title">📚 N-gram Yakalama Dağılımı (df_hits_full — HAM)</div>'))
    _ngram_full_view = _rename_for_report(ng_counts_full, kind="ngram")
    desc = TABLE_DESCRIPTIONS.get("ngram_full", "")
    if desc:
        display(HTML(f"<div class='table-explain'>{desc}</div>"))
    display(HTML(_html_table(_ngram_full_view)))


if "Terim" in _df_hits_full.columns:
    term_summary = (_df_hits_full.groupby("Terim").size()
                    .rename("Adet").reset_index().sort_values(["Adet","Terim"], ascending=[False, True]))
    display(HTML('<div class="sec-title">🗂️ Terim Özeti (df_hits_full — HAM)</div>'))
    _terms_view = _rename_for_report(term_summary.head(30), kind="terms")
    desc = TABLE_DESCRIPTIONS.get("terms_full", "")
    if desc:
        display(HTML(f"<div class='table-explain'>{desc}</div>"))
    display(HTML(_html_table(_terms_view)))

# ============== 9) En negatif / en pozitif cümleler — dual vurgulu ==============
def build_sentence_highlight_df(top_k=5):
    if df_sentence_scores.empty:
        return (pd.DataFrame(columns=["Cümle_ID","Skor","Önizleme (vurgulu)"]),
                pd.DataFrame(columns=["Cümle_ID","Skor","Önizleme (vurgulu)"]))
    tmp = df_sentence_scores if "Cümle_Skor" in df_sentence_scores.columns else df_sentence_scores.assign(Cümle_Skor=0.0)

    prim_by_sid, prim_cls_by_sid, prim_tip_by_sid, \
    extra_by_sid, extra_cls_by_sid, extra_tip_by_sid = _build_span_maps(_df_hits, _df_hits_full)

    rows_pos, rows_neg = [], []
    for _, row in (tmp.sort_values("Cümle_Skor").head(top_k)).iterrows():  # en negatif
        sid = int(row["Cümle_ID"]); text = str(row["Cümle_Orijinal"])
        hl = _highlight_spans(text,
                              prim_by_sid.get(sid, []),  prim_cls_by_sid.get(sid, []),  prim_tip_by_sid.get(sid, []),
                              extra_by_sid.get(sid, []), extra_cls_by_sid.get(sid, []), extra_tip_by_sid.get(sid, []))
        rows_neg.append((sid, float(row["Cümle_Skor"]), hl))
    for _, row in (tmp.sort_values("Cümle_Skor", ascending=False).head(top_k)).iterrows():  # en pozitif
        sid = int(row["Cümle_ID"]); text = str(row["Cümle_Orijinal"])
        hl = _highlight_spans(text,
                              prim_by_sid.get(sid, []),  prim_cls_by_sid.get(sid, []),  prim_tip_by_sid.get(sid, []),
                              extra_by_sid.get(sid, []), extra_cls_by_sid.get(sid, []), extra_tip_by_sid.get(sid, []))
        rows_pos.append((sid, float(row["Cümle_Skor"]), hl))
    df_neg = pd.DataFrame(rows_neg, columns=["Cümle_ID","Skor","Önizleme (vurgulu)"])
    df_pos = pd.DataFrame(rows_pos, columns=["Cümle_ID","Skor","Önizleme (vurgulu)"])
    return df_neg, df_pos

neg_df, pos_df = build_sentence_highlight_df(top_k=5)
display(HTML('<div class="sec-title">⬇️ En Negatif Cümleler</div>'))
display(HTML(neg_df.to_html(escape=False, index=False, classes="compact")))
display(HTML('<div class="sec-title">⬆️ En Pozitif Cümleler</div>'))
display(HTML(pos_df.to_html(escape=False, index=False, classes="compact")))

# ============== 10) Politika satırı hijyen kontrolü (sert) ==============
if not _df_hits_full.empty and "Politika" in _df_hits_full.columns:
    _base = _df_hits_full.copy()
    if "Score_Include" not in _base.columns: _base["Score_Include"] = False
    if "Polarite_Final" not in _base.columns: _base["Polarite_Final"] = 0.0

    _pol_mask = _base["Politika"].map(lambda x: _policy_bucket_strict(x) in ("daraltıcı","genişletici") if pd.notna(x) else False)
    _pol = _base[_pol_mask]

    if not _pol.empty:
        bad_polar = _pol[_pol["Polarite_Final"].fillna(0) != 0]
        bad_incl  = _pol[_pol["Score_Include"].map(_as_bool)] if "Score_Include" in _pol.columns else pd.DataFrame(columns=_pol.columns)

        if len(bad_polar) or len(bad_incl):
            print(f"⚠️ Politika satırı hijyen ihlali: Polarite≠0: {len(bad_polar)} | Score_Include=True: {len(bad_incl)}")
            cols_dbg = [c for c in ["Cümle_ID","Terim","Eşleşme","Politika","Polarite_Final","Score_Include","Regex","Regex_Kaynağı","start","end"] if c in _pol.columns]
            if cols_dbg:
                to_show = pd.concat([bad_polar[cols_dbg], bad_incl[cols_dbg]], axis=0).drop_duplicates()
                display(HTML(_html_table(to_show.sort_values(["Cümle_ID","start"]).head(50))))

# ============== 11) İndirilebilir özet rapor (HTML) ==============
def build_html_report():
    out = StringIO()
    out.write("<html><head><meta charset='utf-8'>")
    out.write("<title>Metin Analizi Özeti</title>")
    out.write(style_css.replace("<style>", "<style> body{font-family:Inter,system-ui,-apple-system,Segoe UI,Roboto; margin:16px;}"))
    out.write(f"""
      <h2>Özet Kartlar</h2>
      {cards_html}
      <h3>Politika Duruşu (Sadece Sözlük Etiketi; HAM)</h3>
      {_html_table(df_policy_counts_strict)}
      <h3>En Negatif Cümleler</h3>
      {neg_df.to_html(escape=False, index=False, classes="compact")}
      <h3>En Pozitif Cümleler</h3>
      {pos_df.to_html(escape=False, index=False, classes="compact")}
    """)
    if isinstance(_df_hits_full, pd.DataFrame) and not _df_hits_full.empty:
        def _ngram_table(df):
            if df is None or df.empty or "Tür" not in df.columns: return None
            return (df["Tür"].value_counts().rename_axis("Tür").reset_index(name="Adet")
                    .sort_values(["Adet","Tür"], ascending=[False, True]))
        ng_counts_full = _ngram_table(_df_hits_full)
        out.write("<h3>N-gram Yakalama Dağılımı (df_hits_full — HAM)</h3>")
        out.write((ng_counts_full.to_html(index=False, classes="compact") if ng_counts_full is not None else "<i>yok</i>"))
        if "Terim" in _df_hits_full.columns:
            term_summary = (_df_hits_full.groupby("Terim").size().rename("Adet")
                            .reset_index().sort_values(["Adet","Terim"], ascending=[False, True]).head(30))
            out.write("<h3>Terim Özeti (df_hits_full — HAM)</h3>")
            out.write(term_summary.to_html(index=False, classes="compact"))
    out.write("</body></html>")
    return out.getvalue()

html_report = build_html_report()
with open("rapor_ozet.html","w",encoding="utf-8") as f:
    f.write(html_report)
print("💾 rapor_ozet.html oluşturuldu (Dosyalar panelinden indirebilirsin).")

# ============== 12) 📜 Tam Metin — Orijinal Sıra + Vurgulu ==============
def build_full_text_highlight_df():
    if df_sentence_scores.empty:
        return pd.DataFrame(columns=["Cümle_ID","Cümle_Skor","Eşleşme_Sayısı","Metin (vurgulu)"])
    prim_by_sid, prim_cls_by_sid, prim_tip_by_sid, \
    extra_by_sid, extra_cls_by_sid, extra_tip_by_sid = _build_span_maps(_df_hits, _df_hits_full)

    rows = []
    for _, r in df_sentence_scores.sort_values("Cümle_ID").iterrows():
        sid   = int(r["Cümle_ID"])
        text  = str(r["Cümle_Orijinal"]).replace("\n", " ")
        score = float(r.get("Cümle_Skor", 0.0))
        cnt   = int(r.get("Eşleşme_Sayısı", 0))
        hl = _highlight_spans(
            text,
            prim_by_sid.get(sid, []),  prim_cls_by_sid.get(sid, []),  prim_tip_by_sid.get(sid, []),
            extra_by_sid.get(sid, []), extra_cls_by_sid.get(sid, []), extra_tip_by_sid.get(sid, [])
        )
        rows.append({"Cümle_ID": sid, "Cümle_Skor": score, "Eşleşme_Sayısı": cnt, "Metin (vurgulu)": hl})
    return pd.DataFrame(rows)

full_df = build_full_text_highlight_df()
display(HTML('<div class="sec-title">📜 Tam Metin (orijinal sıra, vurgulu)</div>'))
display(HTML(full_df.to_html(escape=False, index=False, classes="compact")))

# ---- İndirilebilir tam metin HTML ----
def build_full_text_html(df):
    out = StringIO()
    out.write("<html><head><meta charset='utf-8'>")
    out.write("<title>Tam Metin — Vurgulu</title>")
    out.write(style_css.replace("<style>",
        "<style> body{font-family:Inter,system-ui,-apple-system,Segoe UI,Roboto; margin:16px;} table.compact td, table.compact th{padding:6px 8px; font-size:13px;}"))
    out.write("<h2>📜 Tam Metin (orijinal sıra, vurgulu)</h2>")
    out.write('<table class="compact" style="border-collapse:collapse; width:100%;">')
    out.write('<thead><tr><th style="text-align:left;">ID</th>'
              '<th style="text-align:right;">Skor</th>'
              '<th style="text-align:right;">Eşleşme</th>'
              '<th style="text-align:left;">Metin</th></tr></thead><tbody>')
    for _, row in df.iterrows():
        out.write(
            f"<tr>"
            f"<td>{row['Cümle_ID']}</td>"
            f"<td style='text-align:right;'>{row['Cümle_Skor']:.6f}</td>"
            f"<td style='text-align:right;'>{row['Eşleşme_Sayısı']}</td>"
            f"<td>{row['Metin (vurgulu)']}</td>"
            f"</tr>"
        )
    out.write("</tbody></table></body></html>")
    return out.getvalue()

html_full = build_full_text_html(full_df)
with open("tam_metin_vurgulu.html","w",encoding="utf-8") as f:
    f.write(html_full)
print("💾 tam_metin_vurgulu.html oluşturuldu (Dosyalar panelinden indirebilirsin).")

# ============== 13) Opsiyonel: CSV dışa aktarma ==============
if EXPORT_CSV:
    try:
        if isinstance(_df_hits, pd.DataFrame) and not _df_hits.empty:
            _df_hits.to_csv("hits_tekillestirilmis.csv", index=False, encoding="utf-8")
        if isinstance(_df_hits_full, pd.DataFrame) and not _df_hits_full.empty:
            _df_hits_full.to_csv("hits_tum_HAM.csv", index=False, encoding="utf-8")
        if isinstance(df_sentence_scores, pd.DataFrame) and not df_sentence_scores.empty:
            df_sentence_scores.to_csv("cumle_skorlari.csv", index=False, encoding="utf-8")
        print("💾 CSV dışa aktarımlar tamamlandı (Dosyalar panelinden indirebilirsin).")
    except Exception as _e:
        print("CSV dışa aktarma sırasında hata:", _e)


# ===============================================
# 14) İnteraktif ve İndirilebilir Tablolar (Colab)
# ===============================================

import pandas as pd
from IPython.display import HTML, display, Javascript
from google.colab import files
import io
import base64
from datetime import datetime

# Colab'in kendi tablo widget'ını aç
try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
    COLAB_AVAILABLE = True
except ImportError:
    print("⚠️ Bu hücre Google Colab dışında çalıştırılıyor.")
    COLAB_AVAILABLE = False

# ===============================================
# CSS Stilleri - Modern ve Profesyonel Görünüm
# ===============================================

MODERN_CSS = """
<style>
    .table-container {
        margin: 25px 0;
        padding: 20px;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        border-radius: 12px;
        box-shadow: 0 10px 30px rgba(0,0,0,0.2);
    }
    .table-header {
        background: white;
        padding: 20px;
        border-radius: 8px 8px 0 0;
        margin: -20px -20px 0 -20px;
    }
    .table-title {
        font-size: 22px;
        font-weight: 700;
        color: #2d3748;
        margin: 0 0 10px 0;
        display: flex;
        align-items: center;
        gap: 10px;
    }
    .table-stats {
        display: flex;
        gap: 20px;
        flex-wrap: wrap;
        margin-top: 12px;
    }
    .stat-item {
        background: #f7fafc;
        padding: 8px 16px;
        border-radius: 6px;
        font-size: 13px;
        color: #4a5568;
        border-left: 3px solid #667eea;
    }
    .stat-label {
        font-weight: 600;
        color: #2d3748;
    }
    .table-content {
        background: white;
        padding: 20px;
        border-radius: 0 0 8px 8px;
        margin: 0 -20px -20px -20px;
    }
    .download-buttons {
        display: flex;
        gap: 10px;
        margin-top: 15px;
        padding-top: 15px;
        border-top: 2px solid #e2e8f0;
    }
    .btn-download {
        padding: 10px 20px;
        border: none;
        border-radius: 6px;
        font-weight: 600;
        font-size: 13px;
        cursor: pointer;
        transition: all 0.3s ease;
        display: inline-flex;
        align-items: center;
        gap: 8px;
        text-decoration: none;
    }
    .btn-csv {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
    }
    .btn-csv:hover {
        transform: translateY(-2px);
        box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4);
    }
    .btn-excel {
        background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%);
        color: white;
    }
    .btn-excel:hover {
        transform: translateY(-2px);
        box-shadow: 0 5px 15px rgba(17, 153, 142, 0.4);
    }
    .btn-all {
        background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
        color: white;
    }
    .btn-all:hover {
        transform: translateY(-2px);
        box-shadow: 0 5px 15px rgba(245, 87, 108, 0.4);
    }
    .empty-table {
        text-align: center;
        padding: 40px;
        color: #a0aec0;
        font-style: italic;
        font-size: 14px;
    }
    .timestamp {
        color: #718096;
        font-size: 12px;
        margin-top: 8px;
    }
</style>
"""

display(HTML(MODERN_CSS))

# ===============================================
# Yardımcı Fonksiyonlar
# ===============================================

def create_download_link(df, filename, file_format='csv'):
    """DataFrame için indirme linki oluştur"""
    if df is None or df.empty:
        return ""

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    full_filename = f"{filename}_{timestamp}.{file_format}"

    if file_format == 'csv':
        data = df.to_csv(index=False, encoding='utf-8-sig')
        btn_class = "btn-csv"
        icon = "📄"
    else:  # excel
        buffer = io.BytesIO()
        df.to_excel(buffer, index=False, engine='openpyxl')
        data = buffer.getvalue()
        btn_class = "btn-excel"
        icon = "📊"

    # JavaScript ile indirme tetikle
    b64 = base64.b64encode(data.encode() if isinstance(data, str) else data).decode()
    href = f'data:application/octet-stream;base64,{b64}'

    return f'''
        <a href="{href}" download="{full_filename}" class="btn-download {btn_class}">
            {icon} {file_format.upper()} İndir
        </a>
    '''

def get_table_stats(df):
    """Tablo istatistikleri hesapla"""
    if df is None or df.empty:
        return {}

    stats = {
        'Satır': len(df),
    }

    return stats

def show_interactive_table(df, title, emoji="📊", filename="table", rows_per_page=25):
    """
    Gelişmiş interaktif tablo gösterimi

    Args:
        df: Gösterilecek DataFrame
        title: Tablo başlığı
        emoji: Başlık ikonu
        filename: İndirme dosya adı
        rows_per_page: Sayfa başına satır sayısı
    """

    # Konteyner başlat
    container_html = f'<div class="table-container">'

    # Başlık
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    header_html = f'''
    <div class="table-header">
        <div class="table-title">{emoji} {title}</div>
        <div class="timestamp">🕐 Oluşturulma: {timestamp}</div>
    </div>
    '''

    container_html += header_html

    # Tablo içeriği
    container_html += '<div class="table-content">'

    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        container_html += '<div class="empty-table">📭 Gösterilecek veri bulunamadı</div>'
    else:
        display(HTML(container_html))

        # Interaktif tablo göster
        if COLAB_AVAILABLE:
            try:
                display(data_table.DataTable(df, include_index=False, num_rows_per_page=rows_per_page))
            except:
                display(df.head(rows_per_page))
        else:
            display(df.head(rows_per_page))

        # İndirme butonları
        download_html = '<div class="download-buttons">'
        download_html += create_download_link(df, filename, 'csv')
        download_html += create_download_link(df, filename, 'xlsx')
        download_html += '</div>'

        display(HTML(download_html))
        container_html = ""  # Zaten display ettik

    # Konteyner kapat
    if container_html:
        container_html += '</div></div>'
        display(HTML(container_html))

# ===============================================
# Tüm Tabloları İndirme Fonksiyonu
# ===============================================

def download_all_tables(tables_dict):
    """Tüm tabloları tek bir Excel dosyasında indir"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"tum_tablolar_{timestamp}.xlsx"

    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine='openpyxl') as writer:
        for sheet_name, df in tables_dict.items():
            if df is not None and not df.empty:
                # Sayfa adını Excel uyumlu hale getir (max 31 karakter)
                safe_name = sheet_name[:31].replace('/', '-').replace('\\', '-')
                df.to_excel(writer, sheet_name=safe_name, index=False)

    # İndirmeyi tetikle
    buffer.seek(0)
    files.download(filename)
    print(f"✅ Tüm tablolar '{filename}' olarak indirildi!")

# ===============================================
# 1) Hazırlık: Tabloları Oluştur
# ===============================================

# 1.a) Tutulan eşleşme adedi
_count_df = pd.DataFrame([{
    "Tutulan Eşleşme Adedi": len(_df_hits) if isinstance(_df_hits, pd.DataFrame) else 0
}])

# 1.b) Regex teşhis tablosu (tüm cümleler)
if isinstance(diag_df, pd.DataFrame) and not diag_df.empty:
    cols_diag_all = [
        c for c in ["Cümle_ID","Anchor_Pass","Regex_Tried","Raw_Matches","Cümle_Önizleme"]
        if c in diag_df.columns
    ]
    _diag_all = diag_df.loc[:, cols_diag_all]
    _diag_all = _rename_for_report(_diag_all, kind="diag")
else:
    _diag_all = pd.DataFrame()

# 1.c) Cümle skorları — tümü
cols_sent_all = [
    c for c in [
        "Cümle_ID","Cümle_Orijinal","Cümle_Temiz",
        "Cümle_Skor_RawSum","Eşleşme_Sayısı",
        "Eşleşme_Sayısı_Skor","Cümle_Skor"
    ] if c in df_sentence_scores.columns
]
if cols_sent_all:
    _sent_all = df_sentence_scores.loc[:, cols_sent_all].sort_values("Cümle_ID")
else:
    _sent_all = df_sentence_scores.copy()
_sent_all = _rename_for_report(_sent_all, kind="sentence")

# 1.d) df_hits_full — ham eşleşmeler (tümü)
cols_full_all = [
    c for c in ["Cümle_ID","start","end","Eşleşme","Terim","Tür",
                "Politika","Polarite_Final","Score_Include"]
    if c in _df_hits_full.columns
]
if cols_full_all:
    _full_all = _df_hits_full.loc[:, cols_full_all].sort_values(["Cümle_ID","start"])
else:
    _full_all = _df_hits_full.copy()
_full_all = _rename_for_report(_full_all, kind="hits_full")

# 1.e) df_hits — skora dahil / seçili eşleşmeler (tümü)
cols_hits_all = [
    c for c in ["Cümle_ID","start","end","Eşleşme","Bağlam","Terim","Tür","Kavram",
                "Politika","Regex","Polarite_Base","Polarite_Final",
                "Regex_Kaynağı","Score_Include"]
    if c in _df_hits.columns
]
if cols_hits_all:
    _hits_all = _df_hits.loc[:, cols_hits_all].sort_values(["Cümle_ID","start"])
else:
    _hits_all = _df_hits.copy()
_hits_all = _rename_for_report(_hits_all, kind="hits")

# 1.f) N-gram dağılımları
def _ngram_df(df):
    if df is None or df.empty or "Tür" not in df.columns:
        return pd.DataFrame(columns=["Tür","Adet"])
    return (
        df["Tür"]
        .value_counts()
        .rename_axis("Tür")
        .reset_index(name="Adet")
        .sort_values(["Adet","Tür"], ascending=[False, True])
    )

_ngram_hits = _rename_for_report(_ngram_df(_df_hits), kind="ngram")
_ngram_full = _rename_for_report(_ngram_df(_df_hits_full), kind="ngram")

# 1.g) Terim özetleri
if isinstance(_df_hits, pd.DataFrame) and not _df_hits.empty and "Terim" in _df_hits.columns:
    _terms_hits = (
        _df_hits.groupby("Terim").size()
        .rename("Adet").reset_index()
        .sort_values(["Adet","Terim"], ascending=[False, True])
    )
    _terms_hits = _rename_for_report(_terms_hits, kind="terms")
else:
    _terms_hits = pd.DataFrame(columns=["Terim","Adet"])

if isinstance(_df_hits_full, pd.DataFrame) and not _df_hits_full.empty and "Terim" in _df_hits_full.columns:
    _terms_full = (
        _df_hits_full.groupby("Terim").size()
        .rename("Adet").reset_index()
        .sort_values(["Adet","Terim"], ascending=[False, True])
    )
    _terms_full = _rename_for_report(_terms_full, kind="terms")
else:
    _terms_full = pd.DataFrame(columns=["Terim","Adet"])

# 1.h) Tür × Terim
if isinstance(_df_hits, pd.DataFrame) and not _df_hits.empty and {"Tür","Terim"}.issubset(_df_hits.columns):
    _type_term_hits = (
        _df_hits.groupby(["Tür","Terim"]).size()
        .rename("Adet").reset_index()
        .sort_values(["Tür","Adet","Terim"], ascending=[True, False, True])
    )
    _type_term_hits = _rename_for_report(_type_term_hits, kind="terms")
else:
    _type_term_hits = pd.DataFrame(columns=["Tür","Terim","Adet"])

# ===============================================
# 2) Tabloları Göster
# ===============================================

print("📊 İnteraktif Tablo Sistemi Başlatıldı")
print("=" * 60)

show_interactive_table(_count_df, "Tutulan Eşleşme Adedi", "🔎", "eslesme_adedi")
show_interactive_table(_diag_all, "Regex Teşhis Tablosu — Tümü", "🧪", "regex_teshis")
show_interactive_table(_sent_all, "Cümle Skor Tablosu — Tümü", "🧾", "cumle_skor")
show_interactive_table(_hits_all, "Tutulan Eşleşmeler — df_hits", "✅", "tutulan_eslesmeler")
show_interactive_table(_full_all, "Yakalanan İfadeler — df_hits_full", "🔎", "yakalanan_ifadeler")
show_interactive_table(_ngram_hits, "N-gram Dağılımı — df_hits", "📚", "ngram_hits")
show_interactive_table(_terms_hits, "Terim Özeti — df_hits", "🗂️", "terim_ozeti_hits")
show_interactive_table(_ngram_full, "N-gram Dağılımı — df_hits_full", "📚", "ngram_full")
show_interactive_table(_terms_full, "Terim Özeti — df_hits_full", "🗂️", "terim_ozeti_full")
show_interactive_table(_type_term_hits, "Tür × Terim — df_hits", "🏷️", "tur_terim")

# ===============================================
# 3) Tüm Tabloları Toplu İndirme
# ===============================================

print("\n" + "=" * 60)
print("📦 Toplu İndirme Seçeneği")
print("=" * 60)

all_tables = {
    "Eslesme_Adedi": _count_df,
    "Regex_Teshis": _diag_all,
    "Cumle_Skor": _sent_all,
    "Tutulan_Eslesmeler": _hits_all,
    "Yakalanan_Ifadeler": _full_all,
    "Ngram_Hits": _ngram_hits,
    "Terim_Ozeti_Hits": _terms_hits,
    "Ngram_Full": _ngram_full,
    "Terim_Ozeti_Full": _terms_full,
    "Tur_Terim": _type_term_hits
}

download_btn_html = '''
<div style="text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
     border-radius: 12px; margin: 20px 0;">
    <button onclick="downloadAll()" class="btn-download btn-all"
            style="font-size: 16px; padding: 15px 40px;">
        📦 Tüm Tabloları Tek Dosyada İndir
    </button>
    <div style="color: white; margin-top: 15px; font-size: 13px;">
        Tüm tablolar tek bir Excel dosyasında, ayrı sayfalarda indirilecek
    </div>
</div>
<script>
function downloadAll() {
    google.colab.kernel.invokeFunction('download_all_tables', [], {});
}
</script>
'''

display(HTML(download_btn_html))

# JavaScript callback fonksiyonu kaydet
from google.colab import output
output.register_callback('download_all_tables', lambda: download_all_tables(all_tables))

print("✅ Tüm tablolar hazır!")
print(f"📊 Toplam {len([t for t in all_tables.values() if t is not None and not t.empty])} tablo oluşturuldu.")



Regex sütunu: Regex/Patern | Regex doluluk oranı: 100.00%

🧪 Teşhis Özeti (tüm cümleler):


Cümle No,Anchor Filtresi (Geçti/Kaldı),Denenen Regex Sayısı,Ham Eşleşme Sayısı,Cümle Önizleme
1,21,5,5,Küresel para politikalarına dair belirsizlikler ve küresel büyümeye dair endişeler nedeniyle finans piyasalarındaki oyna
2,21,4,5,2015 yılı Aralık ayında ABD Merkez Bankası (Fed)’nın beklentiler çerçevesinde ilk faiz artışını gerçekleştirmesinin heme
3,21,0,0,"Küresel iktisadi faaliyette 2014 yılından beri yaşanan yavaşlama eğilimi, gelişmekte olan ülkelerde daha belirgin olmak"
4,21,0,0,Emtia fiyatları da yakın dönemde düşüş eğilimini devam ettirmiştir.
5,21,0,0,Gelişmekte olan ülkeler bu dönemde küresel dalgalanmalardan önemli oranda etkilenmiştir.
6,21,1,1,Bu ülkelere yönelik portföy akımları zayıf görünümünü ve kur oynaklıkları ise yüksek seviyelerini sürdürmüştür (Grafik 1
7,21,1,1,Küresel piyasalarda yaşanan oynaklığın etkileri Türkiye ekonomisinde de gözlenmiştir.
8,21,5,5,Bununla birlikte yurt içi belirsizliklerin azalması ve Türkiye Cumhuriyet Merkez Bankası (TCMB) tarafından uygulanmakta
9,21,2,2,Milli gelir ılımlı büyüme eğilimini istikrarlı bir şekilde sürdürmektedir.
10,21,1,1,Artan jeopolitik risklere karşın Avrupa Birliği ülkelerinin talebindeki artışın ihracat üzerindeki olumlu etkisi sürmekt


Politika_Kategori,Adet
daraltıcı,2
genişletici,0
belirsiz/boş,24



🧮 'sıkı para politikası' — STRICT(Terim): 2 satır / 2 cümle | DEBUG(Eşleşme): 2 satır / 2 cümle
   Kovalar (STRICT/sözlük): {'daraltıcı': 2}

🔎 Seçili Eşleşmeler Tablosu (Adım 3-B — Tür Öncelikli Çakışma Çözümü + Adım 2-C — Kapsama Amaçlı Eşleşme Filtresi sonrası)
   Toplam eşleşme satırı: 16


Cümle No,Başlangıç Konumu (Karakter),Bitiş Konumu (Karakter),Metinde Yakalanan İfade,Cümle Bağlamı,Sözlük Terimi,N-gram Türü,Kavram Etiketi,Politika Tonu (Daraltıcı/Genişletici),Kullanılan Regex Deseni,Ham Polarite Puanı,Düzeltilmiş Polarite Puanı,Regex Kaynağı (SAFE/FALLBACK),Skor Hesabına Dahil mi?
1,0,47,Küresel para politikalarına dair belirsizlikler,Küresel para politikalarına dair belirsizlikler ve küresel büyümeye dair endişeler nedeniyle finans\npiyasalarındaki oynaklıklar,küresel belirsizlik,bigram,küresel belirsizlik,,"(?<!\w)küresel[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,8}?(?<!\w)belirsiz[\wçğıöşüÇĞİÖŞÜ]*",-1.0,-1.0,safe,True
1,73,82,endişeler,Küresel para politikalarına dair belirsizlikler ve küresel büyümeye dair endişeler nedeniyle finans\npiyasalarındaki oynaklıklar devam etmektedir.,endişe,unigram,endişe,,(?<!\w)endişe[\wçğıöşüÇĞİÖŞÜ]*(?!\w),-1.0,-1.0,safe,True
1,93,133,finans\npiyasalarındaki oynaklıklar devam,politikalarına dair belirsizlikler ve küresel büyümeye dair endişeler nedeniyle finans\npiyasalarındaki oynaklıklar devam etmektedir.,finansal oynaklık devamı,trigram,finans oynaklık devam,,"(?<!\w)finans[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,8}?(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,8}?(?<!\w)devam[\wçğıöşüÇĞİÖŞÜ]*",-1.0,-1.0,safe,True
2,82,171,faiz artışını gerçekleştirmesinin hemen sonrasında faiz oynaklıklarında bir\nmiktar azalış,"15 yılı Aralık ayında ABD Merkez Bankası (Fed)’nın\nbeklentiler çerçevesinde ilk faiz artışını gerçekleştirmesinin hemen sonrasında faiz oynaklıklarında bir\nmiktar azalış gözlense de, 2016 yılının başından itibaren temelde Çin ekonomisine dair endişe",faiz oynaklıklarında azalış,trigram,faiz oynaklık azalış,,"(?<!\w)faiz[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,10}?\W+(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,10}?\W+(?<!\w)(?:azal(?:ış|ma|d[ıi]|an|[ıi]yor|mak|mek|t[ıi]))[\wçğıöşüÇĞİÖŞÜ]*",1.0,1.0,safe,True
2,245,254,endişeler,"azalış gözlense de, 2016 yılının başından itibaren temelde Çin ekonomisine dair endişeler ve\njeopolitik gelişmeler kaynaklı olarak oynaklıklarda artış yaşanmıştır (Grafi",endişe,unigram,endişe,,(?<!\w)endişe[\wçğıöşüÇĞİÖŞÜ]*(?!\w),-1.0,-1.0,safe,True
2,296,309,oynaklıklarda,temelde Çin ekonomisine dair endişeler ve\njeopolitik gelişmeler kaynaklı olarak oynaklıklarda artış yaşanmıştır (Grafik 1.1).,oynaklık,unigram,oynaklık,,(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?!\w),-1.0,-1.0,safe,True
6,61,73,oynaklıkları,Bu ülkelere\nyönelik portföy akımları zayıf görünümünü ve kur oynaklıkları ise yüksek seviyelerini sürdürmüştür\n(Grafik 1.2).,oynaklık,unigram,oynaklık,,(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?!\w),-1.0,-1.0,safe,True
7,28,38,oynaklığın,Küresel piyasalarda yaşanan oynaklığın etkileri Türkiye ekonomisinde de gözlenmiştir.,oynaklık,unigram,oynaklık,,(?<!\w)oynak[\wçğıöşüÇĞİÖŞÜ]*(?!\w),-1.0,-1.0,fallback,True
8,26,51,belirsizliklerin azalması,Bununla\nbirlikte yurt içi belirsizliklerin azalması ve Türkiye Cumhuriyet Merkez Bankası (TCMB) tarafından\nuygulanmakta olan sıkı p,belirsizlik azalması,bigram,belirsizlik azal,,"(?<!\w)belirsizlik[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,10}?\W+(?<!\w)azal(?:ma|ması|mıştır|dı|dığı|an|ıyor|mak|mektedir|mekte)[\wçğıöşüÇĞİÖŞÜ]*",1.0,1.0,safe,True
8,168,185,finansal istikrar,(TCMB) tarafından\nuygulanmakta olan sıkı para politikası ile diğer likidite ve finansal istikrar politikaları bu etkileri\nsınırlandırmıştır.,finansal istikrar,bigram,finans istikrar,,"(?<!\w)finans[\wçğıöşüÇĞİÖŞÜ]*(?:\W+\w+){0,8}?(?<!\w)istikrar[\wçğıöşüÇĞİÖŞÜ]*",1.0,1.0,safe,True


...

,N-gram Türü,Gözlenen Frekans
0,unigram,14
1,bigram,8
2,trigram,4


,Sözlük Terimi,Metinde Görülme Sayısı
11,oynaklık,5
0,belirsizlik,2
2,endişe,2
3,enflasyon iyileşmesi,2
7,istikrar,2
8,iyileşme,2
14,sıkı para politikası,2
1,belirsizlik azalması,1
4,faiz oynaklıklarında azalış,1
5,finansal istikrar,1


,N-gram Türü,Sözlük Terimi,Metinde Görülme Sayısı
1,bigram,enflasyon iyileşmesi,2
0,bigram,belirsizlik azalması,1
2,bigram,finansal istikrar,1
3,bigram,küresel belirsizlik,1
4,bigram,ılımlı büyüme,1
5,trigram,faiz oynaklıklarında azalış,1
6,trigram,finansal oynaklık devamı,1
7,trigram,sıkı para politikası,1
10,unigram,oynaklık,3
8,unigram,endişe,2



📦 Toplu İndirme Seçeneği


✅ Tüm tablolar hazır!
📊 Toplam 10 tablo oluşturuldu.


In [8]:
# @title
# 🔎 Öz-denetim: mimari garanti testleri

import pandas as pd

errors = []

# 1) df_hits ⊆ df_hits_full (aynı (sid,start,end,Terim,Regex) anahtarına göre)
if 'df_hits' in globals() and 'df_hits_full' in globals() and not df_hits.empty:
    key_cols = ["Cümle_ID","start","end","Terim","Regex"]
    a = df_hits[key_cols].assign(_in_hits=1)
    b = df_hits_full[key_cols].drop_duplicates()
    merged = a.merge(b, on=key_cols, how="left", indicator=True)
    miss = merged[merged["_merge"] == "left_only"]
    if len(miss):
        errors.append(f"df_hits içinde olup df_hits_full içinde olmayan {len(miss)} kayıt var.")
        display(miss.head(10))

# 2) df_hits içinde aynı cümlede (Terim,Regex) tekrarı yok (2-C garantisi)
if 'df_hits' in globals() and not df_hits.empty:
    dup_mask = df_hits.duplicated(subset=["Cümle_ID","Terim","Regex"], keep=False)
    dups = df_hits[dup_mask]
    if len(dups):
        errors.append(f"2-C tekilleştirme ihlali: {len(dups)} satır (aynı Cümle_ID,Terim,Regex).")
        display(dups.sort_values(["Cümle_ID","Terim","Regex"]).head(10))

# 3) df_hits içinde aynı cümlede örtüşen span yok (tür önceliği çözümü)
def _has_overlap(g):
    spans = g[["start","end"]].sort_values("start").to_numpy()
    for i in range(1, len(spans)):
        if spans[i-1][1] > spans[i][0]:
            return True
    return False

if 'df_hits' in globals() and not df_hits.empty:
    bad_sid = []
    for sid, g in df_hits.groupby("Cümle_ID"):
        if _has_overlap(g):
            bad_sid.append(sid)
    if bad_sid:
        errors.append(f"Örtüşme çözümü ihlali: {len(bad_sid)} cümlede çakışan span var. Örnek Cümle_ID: {bad_sid[:5]}")

# 4) Politika tonu satırları skora dahil edilmemiş olmalı (Score_Include=False)
def _bucket(v: str) -> str:
    v = ("" if pd.isna(v) else str(v)).strip().lower()
    if v == "daraltıcı" or "daralt" in v or "sıkı" in v or "sıkılaş" in v: return "daraltıcı"
    if v == "genişletici" or "geniş" in v or "gevş" in v: return "genişletici"
    return ""

if 'df_hits' in globals() and not df_hits.empty and "Politika" in df_hits.columns:
    pol_mask = df_hits["Politika"].map(_bucket).isin({"daraltıcı","genişletici"})
    bad = df_hits.loc[pol_mask & df_hits["Score_Include"].fillna(False)]
    if len(bad):
        errors.append(f"Politika tonu satırı skora dahil edilmiş: {len(bad)} satır.")
        display(bad[["Cümle_ID","Eşleşme","Terim","Politika","Score_Include"]].head(10))

# 5) Skor normalizasyonu paydası = Score_Include True sayısı (n≥0)
if 'df_sentence_scores' in globals() and not df_sentence_scores.empty:
    neg_n = (df_sentence_scores["Eşleşme_Sayısı_Skor"] < 0).sum()
    if neg_n:
        errors.append("Eşleşme_Sayısı_Skor negatif olamaz.")
    # isteğe bağlı: boş cümlede skor 0 olmalı
    zero_ok = df_sentence_scores.loc[df_sentence_scores["Eşleşme_Sayısı_Skor"]==0, "Cümle_Skor"].round(12)==0
    if not zero_ok.all():
        errors.append("Eşleşme_Sayısı_Skor=0 iken Cümle_Skor ≠ 0 tespit edildi.")

print("✅ Öz-denetim tamamlandı." if not errors else "⚠️ Bulgular:")
for e in errors: print(" -", e)


✅ Öz-denetim tamamlandı.


In [9]:
# @title
# ===============================================
# 📊 COLAB HÜCRE — Sunum & Dashboard (Readability Boost)
# Mantık sabit: Sunum, özet ve görselleştirme
# Girdi (motor sonrası): df_hits, df_hits_full, df_sentence_scores, ts_summary_df, (opsiyonel) df_policy_counts
# ===============================================

import pandas as pd
import numpy as np
import math, html, unicodedata
from IPython.display import HTML, display
from io import StringIO

# ---------- Güvenli yardımcılar ----------
def _html_escape(x: str) -> str:
    return html.escape(x or "", quote=False)

def _metric_card(title, value, subtitle=""):
    return f"""
    <div class="card">
      <div class="title">{title}</div>
      <div class="value">{value}</div>
      <div class="sub">{subtitle}</div>
    </div>
    """

def _to_percent(x, digits=1):
    try:
        return f"{100*float(x):.{digits}f}%"
    except Exception:
        return "0%"

def _style_hide_index(df: pd.DataFrame):
    st = df.style.set_table_attributes('class="compact"')
    try:
        return st.hide(axis="index")
    except Exception:
        try:
            return st.hide_index()
        except Exception:
            return st

# ---------- Vurgulama yardımcıları ----------
def _highlight_spans(text, spans, pols):
    if not spans:
        return _html_escape(text)
    pieces, cursor = [], 0
    for (s,e), p in sorted(zip(spans, pols), key=lambda t: t[0][0]):
        if s > cursor:
            pieces.append(_html_escape(text[cursor:s]))
        cls = "pos" if p>0 else ("neg" if p<0 else "neu")
        pieces.append(f'<mark class="{cls}">{_html_escape(text[s:e])}</mark>')
        cursor = e
    pieces.append(_html_escape(text[cursor:]))
    return "".join(pieces)

from collections import defaultdict
def _highlight_spans_dual(text, prim_spans, prim_pols, extra_spans, extra_pols):
    if not text:
        return ""
    kept, occupied = [], []
    def _add_if_free(s, e, cls):
        for s2, e2, _ in occupied:
            if not (e <= s2 or e2 <= s):
                return False
        kept.append((s, e, cls)); occupied.append((s, e, cls)); return True
    for (s, e), p in sorted(zip(prim_spans, prim_pols), key=lambda t: t[0][0]):
        _add_if_free(int(s), int(e), "pos1" if p>0 else ("neg1" if p<0 else "neu"))
    for (s, e), p in sorted(zip(extra_spans, extra_pols), key=lambda t: t[0][0]):
        _add_if_free(int(s), int(e), "pos2" if p>0 else ("neg2" if p<0 else "neu"))
    kept.sort(key=lambda x: x[0])
    pieces, cursor = [], 0
    for s, e, cls in kept:
        if s > cursor:
            pieces.append(_html_escape(text[cursor:s]))
        pieces.append(f'<mark class="{cls}">{_html_escape(text[s:e])}</mark>')
        cursor = e
    pieces.append(_html_escape(text[cursor:]))
    return "".join(pieces)

def _build_dual_span_maps(df_hits, df_hits_full):
    prim_by_sid = defaultdict(list); prim_pols_by_sid = defaultdict(list); viz_keys = set()
    if df_hits is not None and not df_hits.empty:
        for _, r in df_hits.iterrows():
            sid = int(r["Cümle_ID"]); s, e = int(r["start"]), int(r["end"])
            prim_by_sid[sid].append((s, e))
            prim_pols_by_sid[sid].append(float(r["Polarite_Final"]))
            viz_keys.add((sid, s, e, str(r.get("Terim","")), str(r.get("Regex",""))))
    extra_by_sid = defaultdict(list); extra_pols_by_sid = defaultdict(list)
    if df_hits_full is not None and not df_hits_full.empty:
        for _, r in df_hits_full.iterrows():
            sid = int(r["Cümle_ID"]); s, e = int(r["start"]), int(r["end"])
            key = (sid, s, e, str(r.get("Terim","")), str(r.get("Regex","")))
            if key in viz_keys:  # benzersiz görselleştirilmişse atla
                continue
            extra_by_sid[sid].append((s, e))
            extra_pols_by_sid[sid].append(float(r["Polarite_Final"]))
    return prim_by_sid, prim_pols_by_sid, extra_by_sid, extra_pols_by_sid

# ---------- CSS / Lejant ----------
style_css = """
<style>
  .sec-title { font-weight:700; font-size:16px; margin: 14px 0 6px 0; }
  .grid { display:grid; grid-template-columns: repeat(6, minmax(120px,1fr)); gap:12px; margin:12px 0; }
  .card { background:#0b1324; color:#e6edf3; border:1px solid #223; border-radius:14px; padding:12px; box-shadow:0 2px 8px rgba(0,0,0,.18); }
  .card .title { font-size:12px; opacity:.8; }
  .card .value { font-size:22px; font-weight:700; margin-top:4px; }
  .card .sub   { font-size:11px; opacity:.7; margin-top:2px; }
  @media (prefers-color-scheme: light) {
    .card { background:#f8fafc; color:#0f172a; border-color:#e2e8f0; }
  }
  mark.pos1 { background:#86efac; color:#065f46; padding:0 2px; border-radius:4px; }
  mark.neg1 { background:#fca5a5; color:#7f1d1d; padding:0 2px; border-radius:4px; }
  mark.pos2 { background:#dcfce7; color:#065f46; padding:0 2px; border-radius:4px; }
  mark.neg2 { background:#fee2e2; color:#7f1d1d; padding:0 2px; border-radius:4px; }
  mark.neu  { background:#e5e7eb; color:#111827; padding:0 2px; border-radius:4px; }
  table.compact td, table.compact th { padding:6px 8px; font-size:13px; }
  .legend { margin: 8px 0 14px 0; font-size:13px; line-height:1.4; }
  .legend .row { display:flex; gap:14px; flex-wrap:wrap; }
  .legend .item { display:flex; align-items:center; gap:8px; }
  .legend .sw { width:14px; height:14px; border-radius:4px; border:1px solid rgba(0,0,0,.08); }
  .sw.pos1 { background:#86efac; border-color:#22c55e33; }
  .sw.pos2 { background:#dcfce7; border-color:#22c55e33; }
  .sw.neg1 { background:#fca5a5; border-color:#ef444433; }
  .sw.neg2 { background:#fee2e2; border-color:#ef444433; }
</style>
"""

legend_html = """
<div class="sec-title">Metodoloji Özeti — Endeks ve Vurgular</div>
<div class="legend">
  <div class="row">
    <div class="item"><span class="sw neg1"></span> Koyu kırmızı: benzersiz vurguda sayılan <b>negatif</b></div>
    <div class="item"><span class="sw pos1"></span> Koyu yeşil: benzersiz vurguda sayılan <b>pozitif</b></div>
    <div class="item"><span class="sw neg2"></span> Açık kırmızı: <b>sadece endekse</b> eklenen ekstra negatif tekrarlar</div>
    <div class="item"><span class="sw pos2"></span> Açık yeşil: <b>sadece endekse</b> eklenen ekstra pozitif tekrarlar</div>
  </div>
</div>
"""

# ---------- 1) Üst özet kartları ----------
df_sentence_scores = globals().get("df_sentence_scores", pd.DataFrame())
df_hits = globals().get("df_hits", pd.DataFrame())
df_hits_full = globals().get("df_hits_full", pd.DataFrame())
ts_summary_df = globals().get("ts_summary_df", pd.DataFrame())
df_policy_counts = globals().get("df_policy_counts", None)  # motor yaptıysa hazır gelir

doc_sum   = float(df_sentence_scores["Cümle_Skor"].sum()) if len(df_sentence_scores) else 0.0
doc_mean  = float(df_sentence_scores["Cümle_Skor"].mean()) if len(df_sentence_scores) else 0.0
doc_med   = float(df_sentence_scores["Cümle_Skor"].median()) if len(df_sentence_scores) else 0.0
doc_var   = float(df_sentence_scores["Cümle_Skor"].var(ddof=1)) if len(df_sentence_scores)>1 else 0.0

pos_doc = int((df_hits_full["Polarite_Final"] > 0).sum()) if not df_hits_full.empty else 0
neg_doc = int((df_hits_full["Polarite_Final"] < 0).sum()) if not df_hits_full.empty else 0
pn_total = pos_doc + neg_doc

tokens_total = int(
    df_sentence_scores["Cümle_Temiz"].fillna("").str.split().map(len).sum()
) if len(df_sentence_scores) else 0

ts_net      = (pos_doc - neg_doc) / pn_total if pn_total > 0 else 0.0
ts_posshare =  pos_doc / pn_total            if pn_total > 0 else 0.0
ts_net_per1k = 1000.0 * (pos_doc - neg_doc) / max(1, tokens_total)

coverage = float((df_sentence_scores["Eşleşme_Sayısı"]>0).mean()) if len(df_sentence_scores) else 0.0

# ---------- Politika duruşu kartı (fallback üretici) ----------
def _norm_tr(s):
    if s is None: return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("İ","i").replace("I","ı").lower()
    return unicodedata.normalize("NFC", s).strip()

def _policy_bucket(v) -> str:
    t = _norm_tr(v)
    if not t: return "belirsiz/boş"
    if t == "daraltıcı" or "daralt" in t or "sıkı" in t:
        return "daraltıcı"
    if t == "genişletici" or "geniş" in t or "gevş" in t:
        return "genişletici"
    return "belirsiz/boş"

if df_policy_counts is None:
    if not df_hits_full.empty and "Politika" in df_hits_full.columns:
        df_policy_counts = (
            df_hits_full
              .assign(Politika_Kategori=lambda d: d["Politika"].map(_policy_bucket))
              .groupby("Politika_Kategori").size()
              .reindex(["daraltıcı","genişletici","belirsiz/boş"], fill_value=0)
              .reset_index(name="Adet")
        )
    else:
        df_policy_counts = pd.DataFrame({
            "Politika_Kategori": ["daraltıcı","genişletici","belirsiz/boş"],
            "Adet": [0,0,0]
        })

_d = int(df_policy_counts.loc[df_policy_counts["Politika_Kategori"]=="daraltıcı","Adet"].sum())
_g = int(df_policy_counts.loc[df_policy_counts["Politika_Kategori"]=="genişletici","Adet"].sum())
_b = int(df_policy_counts.loc[df_policy_counts["Politika_Kategori"]=="belirsiz/boş","Adet"].sum())

cards_html = f"""
{style_css}
<div class="grid">
  {_metric_card("Doküman Toplam Skor", f"{doc_sum:.4f}", "4-B normalizasyonlu")}
  {_metric_card("Ortalama Cümle Skoru", f"{doc_mean:.4f}", f"Medyan {doc_med:.2f}")}
  {_metric_card("Skor Varyansı", f"{doc_var:.6f}", "Cümle skor dağılımı")}
  {_metric_card("Pozitif/Negatif", f"{pos_doc}/{neg_doc}", f"Toplam eşleşme n={pn_total}")}
  {_metric_card("Net Oran", f"{ts_net:.4f}", f"Pozitif pay {_to_percent(ts_posshare)}")}
  {_metric_card("Net / 1000 kelime", f"{ts_net_per1k:.2f}", f"Toplam kelime {tokens_total}")}
</div>
<div class="grid" style="grid-template-columns: repeat(3, minmax(120px,1fr));">
  {_metric_card("Politika duruşu (adet)", f"D:{_d} | G:{_g} | ?:{_b}", "Kaynak: df_hits_full/Politika")}
  {_metric_card("Kapsama", _to_percent(coverage), "≥1 eşleşme içeren cümle oranı")}
  {_metric_card("N-gram kapsamı",
                ("0" if df_hits.empty else f"{df_hits['Tür'].value_counts().to_dict()}").replace("'", ""),
                "df_hits (tekilleştirilmiş)")}
</div>
"""
display(HTML(cards_html))

display(HTML(legend_html))

# ---------- 2) En negatif / en pozitif cümleler: dual vurgulu ----------
def build_sentence_highlight_df(top_k=5):
    if df_sentence_scores.empty:
        return (pd.DataFrame(columns=["Cümle_ID","Skor","Önizleme (vurgulu)"]),
                pd.DataFrame(columns=["Cümle_ID","Skor","Önizleme (vurgulu)"]))
    prim_by_sid, prim_pols_by_sid, extra_by_sid, extra_pols_by_sid = \
        _build_dual_span_maps(df_hits, df_hits_full)
    rows_pos, rows_neg = [], []
    # en negatif
    for _, row in (df_sentence_scores.sort_values("Cümle_Skor").head(top_k)).iterrows():
        sid = int(row["Cümle_ID"]); text = str(row["Cümle_Orijinal"])
        hl = _highlight_spans_dual(text,
                                   prim_by_sid.get(sid, []),  prim_pols_by_sid.get(sid, []),
                                   extra_by_sid.get(sid, []), extra_pols_by_sid.get(sid, []))
        rows_neg.append((sid, float(row["Cümle_Skor"]), hl))
    # en pozitif
    for _, row in (df_sentence_scores.sort_values("Cümle_Skor", ascending=False).head(top_k)).iterrows():
        sid = int(row["Cümle_ID"]); text = str(row["Cümle_Orijinal"])
        hl = _highlight_spans_dual(text,
                                   prim_by_sid.get(sid, []),  prim_pols_by_sid.get(sid, []),
                                   extra_by_sid.get(sid, []), extra_pols_by_sid.get(sid, []))
        rows_pos.append((sid, float(row["Cümle_Skor"]), hl))
    df_neg = pd.DataFrame(rows_neg, columns=["Cümle_ID","Skor","Önizleme (vurgulu)"])
    df_pos = pd.DataFrame(rows_pos, columns=["Cümle_ID","Skor","Önizleme (vurgulu)"])
    return df_neg, df_pos

neg_df, pos_df = build_sentence_highlight_df(top_k=5)
display(HTML('<div class="sec-title">⬇️ En Negatif Cümleler</div>'))
display(HTML(neg_df.to_html(escape=False, index=False, classes="compact")))
display(HTML('<div class="sec-title">⬆️ En Pozitif Cümleler</div>'))
display(HTML(pos_df.to_html(escape=False, index=False, classes="compact")))

# ---------- 3) N-gram ve Tür×Terim özetleri (bilgi amaçlı) ----------
if not df_hits.empty:
    ng_df = (df_hits.groupby("Tür").size().reset_index(name="Adet")
             .sort_values("Adet", ascending=False).reset_index(drop=True))
    tt_df = (df_hits.groupby(["Tür","Terim"]).size().reset_index(name="Adet")
             .sort_values(["Adet","Tür","Terim"], ascending=[False, True, True]).reset_index(drop=True))
    display(HTML('<div class="sec-title">📚 N-gram Yakalama Dağılımı (df_hits)</div>'))
    display(_style_hide_index(ng_df))
    display(HTML('<div class="sec-title">🏷️ Tür × Terim hit sayıları (azalan)</div>'))
    display(_style_hide_index(tt_df))
else:
    print("ℹ️ df_hits boş: N-gram ve Tür×Terim özetleri gösterilmedi.")

# ---------- 4) En pozitif/negatif terimler (df_hits tabanlı; bilgi amaçlı) ----------
def _term_extremes(df_hits, topn=10):
    if df_hits.empty or "Terim" not in df_hits.columns:
        return (pd.DataFrame(columns=["Terim","Toplam Polarite","Adet"]),
                pd.DataFrame(columns=["Terim","Toplam Polarite","Adet"]))
    gsum = df_hits.groupby("Terim")["Polarite_Final"].sum().rename("Toplam Polarite")
    gcnt = df_hits.groupby("Terim").size().rename("Adet")
    gx = pd.concat([gsum, gcnt], axis=1).reset_index()
    term_pos = gx.sort_values(["Toplam Polarite","Adet"], ascending=[False, False]).head(topn)
    term_neg = gx.sort_values(["Toplam Polarite","Adet"], ascending=[True, False]).head(topn)
    return term_neg, term_pos

term_neg, term_pos = _term_extremes(df_hits, topn=10)

display(HTML('<div class="sec-title">🔻 En Negatif Terimler (bilgi)</div>'))
display(_style_hide_index(term_neg))
display(HTML('<div class="sec-title">🔺 En Pozitif Terimler (bilgi)</div>'))
display(_style_hide_index(term_pos))

# ---------- 5) Politika duruşu tablosu ----------
display(HTML('<div class="sec-title">🧭 Politika Duruşu Dağılımı</div>'))
display(_style_hide_index(df_policy_counts))

# ---------- 6) İndirilebilir özet rapor (HTML) ----------
def build_html_report():
    out = StringIO()
    out.write("<html><head><meta charset='utf-8'>")
    out.write("<title>Metin Analizi Özeti</title>")
    out.write(style_css.replace("<style>","<style> body{font-family:Inter,system-ui,-apple-system,Segoe UI,Roboto; margin:16px;}"))
    out.write(f"""
      <h2>Özet Kartlar</h2>
      {cards_html}
      <h3>Politika Duruşu</h3>
      {df_policy_counts.to_html(index=False, classes="compact")}
      <h3>En Negatif Cümleler</h3>
      {neg_df.to_html(escape=False, index=False, classes="compact")}
      <h3>En Pozitif Cümleler</h3>
      {pos_df.to_html(escape=False, index=False, classes="compact")}
      <h3>En Negatif Terimler (bilgi)</h3>
      {term_neg.to_html(index=False, classes="compact")}
      <h3>En Pozitif Terimler (bilgi)</h3>
      {term_pos.to_html(index=False, classes="compact")}
    """)
    out.write("</body></html>")
    return out.getvalue()

html_report = build_html_report()
with open("rapor_ozet.html","w",encoding="utf-8") as f:
    f.write(html_report)
print("💾 rapor_ozet.html oluşturuldu (Dosyalar panelinden indirebilirsin).")

# ===============================================
# (EK) 📜 Tam Metin — Orijinal Sıra + Vurgulu Çıktı
# Bu bloğu dashboard hücresinin SONUNA ekleyin
# ===============================================

def build_full_text_highlight_df():
    if df_sentence_scores.empty:
        return pd.DataFrame(columns=["Cümle_ID","Cümle_Skor","Eşleşme_Sayısı","Metin (vurgulu)"])
    # KOYU: df_hits (2-C), AÇIK: df_hits_full - df_hits (2-A farkı)
    prim_by_sid, prim_pols_by_sid, extra_by_sid, extra_pols_by_sid = \
        _build_dual_span_maps(df_hits, df_hits_full if 'df_hits_full' in globals() else None)

    rows = []
    for _, r in df_sentence_scores.sort_values("Cümle_ID").iterrows():
        sid   = int(r["Cümle_ID"])
        text  = str(r["Cümle_Orijinal"]).replace("\n", " ")
        score = float(r.get("Cümle_Skor", 0.0))
        cnt   = int(r.get("Eşleşme_Sayısı", 0))
        hl = _highlight_spans_dual(
            text,
            prim_by_sid.get(sid, []),  prim_pols_by_sid.get(sid, []),
            extra_by_sid.get(sid, []), extra_pols_by_sid.get(sid, [])
        )
        rows.append({
            "Cümle_ID": sid,
            "Cümle_Skor": score,
            "Eşleşme_Sayısı": cnt,
            "Metin (vurgulu)": hl
        })
    return pd.DataFrame(rows)

full_df = build_full_text_highlight_df()
display(HTML('<div class="sec-title">📜 Tam Metin (orijinal sıra, vurgulu)</div>'))
full_df_view = _rename_for_report(full_df, kind="fulltext")
desc = TABLE_DESCRIPTIONS.get("fulltext", "")
if desc:
    display(HTML(f"<div class='table-explain'>{desc}</div>"))
display(HTML(full_df_view.to_html(escape=False, index=False, classes="compact")))


# ---- İndirilebilir tam metin HTML ----
from io import StringIO
def build_full_text_html(df):
    out = StringIO()
    out.write("<html><head><meta charset='utf-8'>")
    out.write("<title>Tam Metin — Vurgulu</title>")
    out.write(style_css.replace("<style>",
        "<style> body{font-family:Inter,system-ui,-apple-system,Segoe UI,Roboto; margin:16px;} table.compact td, table.compact th{padding:6px 8px; font-size:13px;}"))
    out.write("<h2>📜 Tam Metin (orijinal sıra, vurgulu)</h2>")
    out.write('<table class="compact" style="border-collapse:collapse; width:100%;">')
    out.write('<thead><tr><th style="text-align:left;">ID</th>'
              '<th style="text-align:right;">Skor</th>'
              '<th style="text-align:right;">Eşleşme</th>'
              '<th style="text-align:left;">Metin</th></tr></thead><tbody>')
    for _, row in df.iterrows():
        out.write(
            f"<tr>"
            f"<td>{row['Cümle_ID']}</td>"
            f"<td style='text-align:right;'>{row['Cümle_Skor']:.6f}</td>"
            f"<td style='text-align:right;'>{row['Eşleşme_Sayısı']}</td>"
            f"<td>{row['Metin (vurgulu)']}</td>"
            f"</tr>"
        )
    out.write("</tbody></table></body></html>")
    return out.getvalue()

html_full = build_full_text_html(full_df)
with open("tam_metin_vurgulu.html","w",encoding="utf-8") as f:
    f.write(html_full)
print("💾 tam_metin_vurgulu.html oluşturuldu (Dosyalar panelinden indirebilirsin).")



Cümle_ID,Skor,Önizleme (vurgulu)
1,-1.429516,Küresel para politikalarına dair belirsizlikler ve küresel büyümeye dair endişeler nedeniyle finans\npiyasalarındaki oynaklıklar devam etmektedir.
6,-1.000000,Bu ülkelere\nyönelik portföy akımları zayıf görünümünü ve kur oynaklıkları ise yüksek seviyelerini sürdürmüştür\n(Grafik 1.2).
7,-1.000000,Küresel piyasalarda yaşanan oynaklığın etkileri Türkiye ekonomisinde de gözlenmiştir.
10,-1.000000,Artan jeopolitik risklere\nkarşın Avrupa Birliği ülkelerinin talebindeki artışın ihracat üzerindeki olumlu etkisi sürmektedir.
2,-0.476505,"2015 yılı Aralık ayında ABD Merkez Bankası (Fed)’nın\nbeklentiler çerçevesinde ilk faiz artışını gerçekleştirmesinin hemen sonrasında faiz oynaklıklarında bir\nmiktar azalış gözlense de, 2016 yılının başından itibaren temelde Çin ekonomisine dair endişeler ve\njeopolitik gelişmeler kaynaklı olarak oynaklıklarda artış yaşanmıştır (Grafik 1.1)."


Cümle_ID,Skor,Önizleme (vurgulu)
9,1.181232,Milli gelir ılımlı büyüme eğilimini istikrarlı bir şekilde sürdürmektedir.
8,1.181232,Bununla\nbirlikte yurt içi belirsizliklerin azalması ve Türkiye Cumhuriyet Merkez Bankası (TCMB) tarafından\nuygulanmakta olan sıkı para politikası ile diğer likidite ve finansal istikrar politikaları bu etkileri\nsınırlandırmıştır.
12,1.000000,"TCMB,\nenflasyon görünümünde belirgin bir iyileşme sağlanana kadar sıkı para politikası duruşunu sürdürecektir."
11,1.000000,"Enflasyon\ngelişmeleri değerlendirildiğinde, enerji fiyatlarındaki gelişmeler enflasyonu olumlu yönde etkilemeye\ndevam ederken artan maliyet unsurları çekirdek enflasyon eğilimindeki iyileşmeyi sınırlamaktadır."
5,0.000000,Gelişmekte olan ülkeler bu dönemde küresel dalgalanmalardan önemli oranda etkilenmiştir.


Tür,Adet
unigram,7
bigram,6
trigram,3


Tür,Terim,Adet
unigram,oynaklık,3
bigram,enflasyon iyileşmesi,2
unigram,endişe,2
bigram,belirsizlik azalması,1
bigram,finansal istikrar,1
bigram,küresel belirsizlik,1
bigram,ılımlı büyüme,1
trigram,faiz oynaklıklarında azalış,1
trigram,finansal oynaklık devamı,1
trigram,sıkı para politikası,1


Terim,Toplam Polarite,Adet
oynaklık,-3.000000,3
endişe,-2.000000,2
finansal oynaklık devamı,-1.000000,1
küresel belirsizlik,-1.000000,1
risk,-1.000000,1
sıkı para politikası,0.000000,1
belirsizlik azalması,1.000000,1
faiz oynaklıklarında azalış,1.000000,1
finansal istikrar,1.000000,1
istikrar,1.000000,1


Terim,Toplam Polarite,Adet
enflasyon iyileşmesi,2.000000,2
belirsizlik azalması,1.000000,1
faiz oynaklıklarında azalış,1.000000,1
finansal istikrar,1.000000,1
istikrar,1.000000,1
ılımlı büyüme,1.000000,1
sıkı para politikası,0.000000,1
finansal oynaklık devamı,-1.000000,1
küresel belirsizlik,-1.000000,1
risk,-1.000000,1


Politika_Kategori,Adet
daraltıcı,2
genişletici,0


💾 rapor_ozet.html oluşturuldu (Dosyalar panelinden indirebilirsin).


Cümle No,Normalize Cümle Skoru,Toplam Eşleşme Sayısı,Metin (vurgulu)
1,-1.429516,3,Küresel para politikalarına dair belirsizlikler ve küresel büyümeye dair endişeler nedeniyle finans piyasalarındaki oynaklıklar devam etmektedir.
2,-0.476505,3,"2015 yılı Aralık ayında ABD Merkez Bankası (Fed)’nın beklentiler çerçevesinde ilk faiz artışını gerçekleştirmesinin hemen sonrasında faiz oynaklıklarında bir miktar azalış gözlense de, 2016 yılının başından itibaren temelde Çin ekonomisine dair endişeler ve jeopolitik gelişmeler kaynaklı olarak oynaklıklarda artış yaşanmıştır (Grafik 1.1)."
3,0.000000,0,"Küresel iktisadi faaliyette 2014 yılından beri yaşanan yavaşlama eğilimi, gelişmekte olan ülkelerde daha belirgin olmak üzere, 2015 yılı ikinci yarısında sürmüştür."
4,0.000000,0,Emtia fiyatları da yakın dönemde düşüş eğilimini devam ettirmiştir.
5,0.000000,0,Gelişmekte olan ülkeler bu dönemde küresel dalgalanmalardan önemli oranda etkilenmiştir.
6,-1.000000,1,Bu ülkelere yönelik portföy akımları zayıf görünümünü ve kur oynaklıkları ise yüksek seviyelerini sürdürmüştür (Grafik 1.2).
7,-1.000000,1,Küresel piyasalarda yaşanan oynaklığın etkileri Türkiye ekonomisinde de gözlenmiştir.
8,1.181232,2,Bununla birlikte yurt içi belirsizliklerin azalması ve Türkiye Cumhuriyet Merkez Bankası (TCMB) tarafından uygulanmakta olan sıkı para politikası ile diğer likidite ve finansal istikrar politikaları bu etkileri sınırlandırmıştır.
9,1.181232,2,Milli gelir ılımlı büyüme eğilimini istikrarlı bir şekilde sürdürmektedir.
10,-1.000000,1,Artan jeopolitik risklere karşın Avrupa Birliği ülkelerinin talebindeki artışın ihracat üzerindeki olumlu etkisi sürmektedir.


💾 tam_metin_vurgulu.html oluşturuldu (Dosyalar panelinden indirebilirsin).


In [10]:
# @title
import pandas as pd
import html as _html2
from io import StringIO as _StringIO

def build_interactive_full_html_report():
    """
    df_sentence_scores, _df_hits, _df_hits_full, diag_df vb. zaten Hücre-5/14'te
    üretilmiş ve _rename_for_report ile insan-dostu kolon adları verilmiş durumda.
    Burada bunları tek bir interaktif HTML raporda topluyoruz.
    """

    # --- 1) Tablo listesi (başlık, dataframe, HTML id) ---
    tables = [
        ("🔎 Tutulan eşleşme adedi (toplam)",        _count_df,       "tbl_count"),
        ("🧪 Regex Teşhis Tablosu — Tümü",           _diag_all,       "tbl_diag_all"),
        ("🧾 Cümle Skor Tablosu (Tümü)",             _sent_all,       "tbl_sent_all"),
        ("✅ Tutulan Eşleşmeler — df_hits (Tümü)",   _hits_all,       "tbl_hits_all"),
        ("🔎 Yakalanan İfadeler — df_hits_full",     _full_all,       "tbl_full_all"),
        ("📚 N-gram Yakalama Dağılımı — df_hits",    _ngram_hits,     "tbl_ngram_hits"),
        ("🗂️ Terim Özeti — df_hits",                _terms_hits,     "tbl_terms_hits"),
        ("📚 N-gram Yakalama Dağılımı — df_hits_full", _ngram_full,   "tbl_ngram_full"),
        ("🗂️ Terim Özeti — df_hits_full",           _terms_full,     "tbl_terms_full"),
        ("🏷️ Tür × Terim — df_hits",                _type_term_hits, "tbl_type_term_hits"),
    ]

    out = _StringIO()
    out.write("<html><head><meta charset='utf-8'>")
    out.write("<title>Metin Analizi — İnteraktif Tablolar</title>")

    # --- 2) DataTables CSS/JS (CDN) ---
    out.write("""
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.8/css/jquery.dataTables.min.css">
<link rel="stylesheet" href="https://cdn.datatables.net/buttons/2.4.2/css/buttons.dataTables.min.css">
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.8/js/jquery.dataTables.min.js"></script>
<script src="https://cdn.datatables.net/buttons/2.4.2/js/dataTables.buttons.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/jszip/3.10.1/jszip.min.js"></script>
<script src="https://cdn.datatables.net/buttons/2.4.2/js/buttons.html5.min.js"></script>
<style>
  body { font-family: Inter, system-ui, -apple-system, Segoe UI, Roboto; margin:16px; }
  table.dataTable.compact thead th { white-space: nowrap; }
</style>
""")

    out.write("</head><body>")
    out.write("<h1>Metin Analizi — İnteraktif Tablolar</h1>")

    # --- 3) Tabloları HTML olarak gövdeye yaz ---
    for title, df, tid in tables:
        if df is None or not isinstance(df, pd.DataFrame) or df.empty:
            continue
        out.write(f"<h2>{_html2.escape(title)}</h2>")
        df2 = df.copy()
        # NaN temizliği
        for c in df2.columns:
            df2[c] = df2[c].map(lambda v: "" if pd.isna(v) else v)
        out.write(df2.to_html(
            index=False,
            escape=True,
            classes="display compact",
            table_id=tid
        ))

    # --- 4) DataTables init script'i ---
    out.write("<script>")
    out.write("$(document).ready(function(){\n")
    out.write("  var commonOpts = {\n"
              "    dom: 'Bfrtip',\n"
              "    buttons: ['copyHtml5','csvHtml5','excelHtml5'],\n"
              "    pageLength: 25,\n"
              "    lengthMenu: [[25,50,100,-1],[25,50,100,'Tümü']],\n"
              "    order: [],\n"
              "    deferRender: true,\n"
              "    scrollX: true\n"
              "  };\n")
    for _, _, tid in tables:
        out.write(f"  if (document.getElementById('{tid}')) "
                  f"$('#{tid}').DataTable(commonOpts);\n")
    out.write("});")
    out.write("</script>")

    out.write("</body></html>")
    return out.getvalue()

# --- 5) HTML dosyasını üret ---
html_dt = build_interactive_full_html_report()
with open("rapor_interaktif_tam.html","w",encoding="utf-8") as f:
    f.write(html_dt)

print("💾 rapor_interaktif_tam.html oluşturuldu (Dosyalar panelinden indirip tarayıcıda aç).")


💾 rapor_interaktif_tam.html oluşturuldu (Dosyalar panelinden indirip tarayıcıda aç).


In [11]:
# @title
